# Partición, tipo de cambio, pruebas con Haiku y evaluación del buscador

## Contexto y objetivo
Este notebook corresponde a la segunda etapa de un proyecto de fine-tuning con QLoRA (Unsloth) de un modelo Qwen de tamaño reducido. El sistema objetivo es un agente que responde preguntas en lenguaje natural consultando la API de Series de Tiempo de la República Argentina. La etapa anterior (notebook 00) produjo un catálogo verificado contra la API. Esta etapa construye y evalúa la capa de búsqueda sobre ese catálogo y fija la partición de datos que se usará en el entrenamiento y la evaluación.

Los resultados de este documento corresponden a datos consultados en noviembre de 2026. La API, el catálogo y los precios del modelo pueden cambiar con el tiempo.

## Resumen de resultados
- La partición por `dataset_id` dejaba familias de datasets casi idénticas repartidas entre splits (139 de las 673 series de test, 20,7%). Se reemplazó por una partición por familias de datasets: 136 datasets agrupados en 97 familias.
- Series usables por split: 5.696 (train), 417 (validación) y 408 (test). Con la incorporación del tipo de cambio, train pasa a 5.704.
- El buscador (SQLite FTS5 con prefijos) alcanza en validación un recall@5 de 86,2% y un recall@10 de 94,7% (n = 94).
- Se incorporó al catálogo el dataset de tipo de cambio, que había quedado fuera del recorte original. Quedó en train, sin evaluación held-out.
- Se verificó el acceso a Claude Haiku 5.5 y se midió el efecto de desactivar su razonamiento en la generación de preguntas.

## Contenido del notebook
1. Partición por familias de datasets (versión final, congelada).
2. Incorporación del dataset de tipo de cambio al catálogo y al índice.
3. Verificación de acceso a Claude Haiku 5.5 y prueba de paráfrasis.
4. Evaluación del buscador sobre el conjunto de desarrollo (95 preguntas de series de test) y comparación de variantes de consulta.
5. Prueba de razonamiento activado y desactivado en la generación de preguntas (20 series de train).
6. Evaluación final del buscador sobre validación (94 preguntas, una sola corrida).

## Partición por familias de datasets

### Motivo
La partición inicial asignaba cada `dataset_id` a un único split. Sin embargo, el portal separa en datasets distintos series de una misma familia (por ejemplo, los IPC provinciales, o los distintos ámbitos del esquema ahorro-inversión-financiamiento). Una comparación de títulos mostró que 7 datasets de test pertenecían a familias que cruzaban splits, con 139 de 673 series de test afectadas. La cifra es un piso, porque el umbral de similitud dejaba fuera familias reales.

### Método
- Las familias se arman con tres reglas: similitud de título mayor o igual a 0,85, misma primera frase del título y uniones manuales documentadas en el código.
- Se evalúan 5.000 repartos aleatorios (cada tema por separado, con ruido en el orden de asignación) y se conserva el de menor puntaje. El puntaje es una heurística propia: suma de desvíos relativos respecto de la proporción 80/10/10 en tema, frecuencia y estado, con un tope de 150 series por familia y un piso de 100 en el denominador. Sirve para comparar repartos, no tiene interpretación absoluta.
- El resultado es determinístico (semilla 3479, puntaje 0,689). El script verifica que coincida con el resultado revisado antes de guardar.

### Resultado
- 136 datasets, 97 familias, 15 de ellas con más de un dataset.
- Series usables: 5.696 / 417 / 408. Con el tope de 150 por familia aplicado al balance: 3.396 / 417 / 408.
- Los seis temas están representados en los tres splits.

### Alternativa descartada
Se evaluó forzar un mínimo de 30 series de Precios en test mediante una penalización y una búsqueda local. El método movió la familia del IPC completa (609 series) al test, dejando al test con 61% de series de un único tema, a train sin el IPC y a validación sin Precios. Se descartó y se mantuvo la partición original.

### Limitaciones de la partición
- Precios tiene 15 series en test: no permite conclusiones por tema.
- No hay series semestrales en test ni diarias en validación. Test tiene 6 series diarias.
- Las tres series `sin_fecha` quedaron en train, por lo que esa rareza de la API no se mide en test.
- Algunos pares de datasets parecidos se mantuvieron separados a propósito, por no tratarse del mismo contenido según sus títulos: 305/443, 41/453, 330/359, 370/380, 323/324 y 112/23. Los títulos están truncados en las impresiones, por lo que la decisión se tomó con información parcial.
- La familia con mayor peso en train (IPC GBA y Nacional, 609 series usables) concentra el 75% de Precios en train. En test, Precios evalúa por lo tanto otros índices (provinciales, mayoristas y canastas).

## Incorporación del tipo de cambio

### Motivo
El tema "Mercados Financieros" no formaba parte del recorte original, de modo que la serie de tipo de cambio del BNA (`168.1_T_CAMBIOR_D_0_0_26`) no estaba en el índice. Las búsquedas de "tipo de cambio" devolvían únicamente índices de tipo de cambio real.

### Procedimiento
Se incorporó únicamente el dataset "Tipo de Cambio $-USD - Futuro Dólar" (19 series en el catálogo, 2 discontinuadas excluidas). Las 17 restantes se verificaron contra la API con el mismo método de la etapa anterior. Se asignaron a train con una familia propia, sin modificar la partición congelada, y se reconstruyó el índice.

### Resultado
- Estados de las 17 series: 6 `normal`, 2 `api_adelantada`, 4 `api_truncada` y 5 `sin_datos`. Son usables 8 de 17; 9 (53%) no lo son.
- El catálogo pasa de 8.209 a 8.226 series. El recall sobre las 95 preguntas de desarrollo no cambió tras la reconstrucción del índice.
- El dataset mezcla tipo de cambio spot (BNA vendedor, MAE, A3500, ADRs), futuros ROFEX de 1 a 6 meses, interés abierto y volumen, todos bajo la misma `distribucion_titulo`. La unidad de las series de interés abierto figura como "Tasa de Interés", lo que parece una etiqueta incorrecta; no se verificó.
- Las consultas más naturales ("tipo de cambio", "dólar") devuelven con frecuencia series truncadas, cuyos datos terminan antes de lo que declara el catálogo.

### Limitación
El tipo de cambio no tiene evaluación held-out, porque se asignó a train.

## Buscador

### Diseño
- Índice SQLite FTS5 sobre las 8.226 series, incluidas las vacías y truncadas, para que el agente pueda encontrarlas y se puedan evaluar los casos sin datos.
- Texto indexado: el título de la serie (columna propia) y un contexto que concatena título de la distribución, descripción, título del dataset, tema, unidades y frecuencia. No se utilizan campos numéricos del catálogo, por los errores detectados en la etapa anterior.
- Ranking BM25 con ponderación 2,0 para el título y 1,0 para el contexto.
- Consulta: palabras sin acentos ni signos, sin una lista de palabras vacías (artículos, preposiciones y similares), unidas con OR. En la variante elegida, las palabras de seis o más letras se reemplazan por sus cinco primeros caracteres con comodín.

### Evaluación
Las preguntas se generaron con Claude Haiku 5.5 a partir de los metadatos de una serie objetivo, con la consigna de no copiar el título técnico, no incluir fechas ni números y distinguir series parecidas. La métrica es el recall@k de la serie exacta.

**Conjunto de desarrollo (test, n = 95)**

| Variante | @1 | @3 | @5 | @10 |
|---|---|---|---|---|
| Palabras exactas | 46,3% | 71,6% | 75,8% | 85,3% |
| Prefijos | 57,9% | 75,8% | 86,3% | 91,6% |
| Prefijos y lista de palabras genéricas | 61,1% | 77,9% | 86,3% | 90,5% |

Se adoptó la variante de prefijos. La lista de palabras genéricas no mostró un beneficio distinguible y se construyó observando los fallos de este mismo conjunto.

**Validación (n = 94, variante de prefijos, una sola corrida)**

| | Recall | IC 95% (Wilson) |
|---|---|---|
| @1 | 58,5% | 48% a 68% |
| @3 | 76,6% | 67% a 84% |
| @5 | 86,2% | 78% a 92% |
| @10 | 94,7% | 88% a 98% |

Con k = 10, la serie correcta está entre los candidatos en 94,7% de los casos. Se propone devolver 10 resultados en la herramienta de búsqueda; la decisión puede revisarse con la evaluación del agente.

### Fallos observados
Los tipos de error identificados provienen de una muestra de fallos, no de un conteo exhaustivo:
- Series hermanas que se distinguen solo por un plazo, una moneda o un sector (por ejemplo, plazos fijos de distintos vencimientos).
- Negaciones en la pregunta, que un buscador por palabras no interpreta.
- Preguntas que omiten un detalle de base (por ejemplo, la base de un indicador disponible en más de una).
- Sinónimos, siglas y variantes de palabra (por ejemplo, "autos" y "automóviles", "Aduana" y "dga").

## Generación de preguntas con Claude Haiku 5.5

### Acceso y costo
- Modelo `claude-haiku-5-5`. Tarifas consultadas: USD 0,10 por millón de tokens de entrada y USD 0,50 de salida para prompts de hasta 100.000 tokens.
- Costo medido: unos USD 0,0002 por pregunta (USD 0,0207 por 94 preguntas), con razonamiento activado.

### Razonamiento
El modelo razona por defecto, y los tokens de razonamiento se facturan como salida. Con `max_tokens` de 400, 31 de 95 generaciones fallaron porque el razonamiento consumió el límite antes de producir texto. Con `max_tokens` de 2.000 no hubo fallos.

Prueba con 20 series de train (las mismas en ambos modos):

| | Con razonamiento | Sin razonamiento |
|---|---|---|
| Tokens de salida (promedio) | 340 | 78 |
| Costo por pregunta | USD 0,00020 | USD 0,00007 |
| Palabras por pregunta | 26,6 | 28,9 |
| Preguntas con dígitos | 0% | 25% |
| Copian el título técnico | 5% | 5% |
| Recall@5 | 95% | 100% |

Desactivar el razonamiento (`thinking: {"type": "disabled"}`) funciona en la cuenta utilizada y reduce el costo unas 2,8 veces, pero 25% de las preguntas incluyen dígitos copiados de los metadatos, en contra de la consigna. La diferencia de recall equivale a una pregunta y no es interpretable con n = 20. **Decisión: mantener el razonamiento activado** para la generación del dataset.

### Cambios de sentido
En una inspección de 6 pares se observaron paráfrasis que cambian el sentido de la pregunta (por ejemplo, invertir qué entidad compra divisas a cuál) o que agregan detalles ausentes de los metadatos. Los filtros automáticos de período, indicador y tipo de agregación no detectan este tipo de error. Antes de entrenar se revisará manualmente una muestra.

## Notas metodológicas y correcciones
- Una estimación inicial de costo (USD 0,15 por 5.000 paráfrasis) no consideraba los tokens de razonamiento; la medición indica un orden de USD 1.
- La hipótesis de que las generaciones fallidas correspondían a preguntas más difíciles no se sostuvo: el recall@5 fue 75% en la primera tanda y 77% en el reintento.
- Una comprobación inicial sobre exportaciones de soja era demasiado restrictiva y no permitía concluir; no se resolvió si existen exportaciones de soja en el recorte con otro nombre.
- Se anticipó que la consulta "cotización del dólar" fallaría por vocabulario ausente; no falló, porque las palabras sin coincidencia se ignoran al unir con OR.
- El puntaje de la partición no es comparable entre versiones, porque su definición cambió entre ellas.
- La clave de la API caducó durante la evaluación en validación (error 401 desde la pregunta 69 de 94). La corrida se retomó con una clave nueva, regenerando solo las preguntas fallidas con el mismo prompt y modelo. Un error de la primera versión del script, que contaba como realizadas las filas fallidas, se corrigió antes de retomar.
- La comparación de variantes del buscador se hizo sobre el conjunto de test, observando sus propios fallos. La validación se utilizó una sola vez, para el número final.

## Limitaciones
- Las preguntas las genera un modelo a partir de los mismos metadatos que alimentan el índice, por lo que el recall es optimista respecto de usuarios reales. Tienen unas 28 palabras en promedio.
- Los intervalos de confianza suponen independencia entre preguntas. Las 94 de validación provienen de 17 datasets y 17 familias, por lo que el margen real es mayor.
- El recall exige la serie exacta, por lo que subestima la utilidad cuando el buscador devuelve series hermanas igualmente válidas.
- Los desgloses por tema y por frecuencia tienen muy pocas preguntas y son solo orientativos. Dinero y Bancos (recall@5 de 50% con n = 12) probablemente corresponde a un único dataset.
- Las comparaciones de la prueba de razonamiento usan n = 20.
- No se evaluó el efecto de las paráfrasis sobre el sentido de las preguntas más allá de la inspección de 6 pares.

## Advertencias de ejecución
- Las celdas de partición y de incorporación del tipo de cambio **escriben archivos en Drive**. Ambas tienen la variable `SAVE`, que debe permanecer en `False`. La de partición verifica que el resultado coincida con el revisado; la del tipo de cambio falla si las series ya están en el catálogo. Ejecutar el guardado una segunda vez sobrescribiría el respaldo con el archivo ya modificado.
- Las celdas que usan la API de Claude requieren la clave en los Secrets de Colab (`ANTHROPIC_API_KEY`) con el acceso habilitado para el notebook. La clave no debe escribirse en el código ni imprimirse.

## Archivos generados (Drive: `qlora-series-ar/data/`)
- `catalogo_busqueda.csv`: 8.226 series con columnas `familia`, `split` y texto de búsqueda.
- `particion_datasets.csv`: asignación de cada dataset y familia a un split.
- `busqueda.db`: índice SQLite FTS5.
- `catalogo_busqueda_respaldo.csv`: estado previo a la partición por familias.
- `catalogo_busqueda_antes_tc.csv` y `busqueda_antes_tc.db`: estado previo a incorporar el tipo de cambio.
- `verificacion_tipo_cambio.csv`: verificación de las 17 series de tipo de cambio.
- `eval_busqueda_preguntas.csv` y `eval_busqueda_preguntas_v2.csv`: preguntas del conjunto de desarrollo (64 y 95).
- `eval_busqueda_val.csv`: preguntas y rangos de la evaluación en validación (94).
- `prueba_razonamiento.csv`: resultados de la comparación con y sin razonamiento.
- `*_parcial.csv`: archivos de retoma, que pueden eliminarse.

## Reproducción
Ejecutar las celdas en orden con `SAVE = False`. La partición y el índice se leen desde los archivos guardados. Las preguntas generadas con el modelo no son deterministas y dependen del estado de la API y del modelo, por lo que los resultados de las celdas que usan Haiku pueden diferir ligeramente al repetirse.

## Próximos pasos
1. Plantillas de preguntas por tipo, con la consulta correcta calculada por código, y caché de pedidos a la API con registro de fallos.
2. Trayectorias de entrenamiento (búsqueda, consulta y respuesta) con resultados reales de las herramientas.
3. Paráfrasis con filtros automáticos y revisión manual de una muestra.
4. Conjunto de preguntas escritas a mano, sin pasar por un modelo, para medir el buscador y el agente.
5. Agente, observabilidad con Langfuse, línea base, fine-tuning y evaluación comparativa.
6. Saneamiento del código y migración a módulos antes de publicar el repositorio.
7. Confirmación de la licencia de los datos y de los términos de uso de las salidas del modelo generador.

## Referencias
- Precios de la API de Claude: https://platform.claude.com/docs/en/about-claude/pricing
- Documentación de la API de Series de Tiempo: https://datosgobar.github.io/series-tiempo-ar-api/
- Documentación de Anthropic sobre el parámetro `thinking` (sección de resolución de problemas), en platform.claude.com/docs.

---
**Ficha**

| | |
|---|---|
| **Runtime** | CPU (sin GPU) |
| **Entradas** (`MyDrive/qlora-series-ar/data/`) | `catalogo_completo.csv`, `catalogo_verificado.csv`, caché de la API (`cache_api.db`) |
| **Salidas principales** | `catalogo_busqueda.csv`, `specs_train_v1.csv`, `specs_val_v2.csv`, `specs_test_v2.csv`, `train_final2.csv`, `eval_val_final2.csv`, `eval_test_final2.csv` y archivos de verificación |
| **Secretos de Colab** | `ANTHROPIC_API_KEY` |
| **Costo y tiempo** | Las celdas que llaman a Claude Haiku 5.5 imprimen su costo; los valores impresos van de USD 0,00 a 0,25 por celda |
| **Orden** | Después de `00`; antes de `01`. Es un cuaderno de trabajo: las celdas están en el orden en que se hicieron |
| **En el repositorio** | Migrado en parte: `particion.py`, `herramientas.py` y `scripts/particionar.py`. La generación de preguntas y los validadores siguen en este cuaderno |

**Advertencias.** Varias celdas llaman a la API de pago de Anthropic. Las celdas están en orden cronológico, y las posteriores corrigen a las anteriores (por ejemplo, las que generan los archivos `final2`).


## Estado de la migración
Este notebook es el registro cronológico de cómo se construyeron las especificaciones y las preguntas: las correcciones están en el orden en que se hicieron, y por eso hay celdas que reemplazan lo que producían las anteriores.

La versión final de las reglas vive en `src/series_qlora/` (`cache_api.py`, `especificaciones.py`, `preguntas.py`) y se ejecuta con `scripts/preparar_preguntas.py`. Para rehacer los conjuntos conviene usar ese script, no este notebook. No está migrada la evaluación del buscador (recall@k) de las celdas iniciales.

In [ ]:
!pip -q install anthropic

import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

message = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": "Hello, world"}],
)

print(message.content[0].text)
print(message.usage)

In [ ]:
import json

pregunta = "¿Cuál fue el tipo de cambio promedio de marzo de 2024?"
prompt = f"""Reescribí esta pregunta de 3 maneras distintas: una formal, una coloquial y una corta.
No cambies el año, el período ni el indicador.
Devolvé solo una lista JSON de 3 strings.

Pregunta: {pregunta}"""

msg = client.messages.create(
    model="claude-haiku-5-5",
    max_tokens=500,
    messages=[{"role": "user", "content": prompt}],
)

texto = msg.content[0].text
print(texto)

# ¿Es JSON válido? ¿Conservan el año y el período?
try:
    lista = json.loads(texto)
    for p in lista:
        print("OK año/mes:", "2024" in p and "marzo" in p.lower(), "|", p)
except json.JSONDecodeError:
    print("\nNo vino como JSON limpio: habrá que ajustar el prompt o limpiar la salida.")

costo = msg.usage.input_tokens * 0.10 / 1e6 + msg.usage.output_tokens * 0.50 / 1e6
print(f"\nTokens: {msg.usage.input_tokens} entrada / {msg.usage.output_tokens} salida -> ${costo:.6f}")

In [ ]:
!pip -q install anthropic
import os, re, json, sqlite3, shutil, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import anthropic
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive, userdata

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"

df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
assert "familia" in df.columns, "Falta la columna 'familia': la partición por familias no está guardada"
meta = pd.read_csv(f"{RUTA}/catalogo_completo.csv", low_memory=False)
print("Usables por split:", df[df["estado"].isin(["normal", "api_adelantada"])]["split"].value_counts().to_dict())

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=6)
MODEL = "claude-haiku-5-5"

# ---------- Buscador (índice FTS5 guardado) ----------
DB = "/content/busqueda.db"
if not os.path.exists(DB):
    shutil.copy(f"{RUTA}/busqueda.db", DB)
con = sqlite3.connect(DB)

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}

def buscar_ids(q, k=10):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    if not toks:
        return []
    match = " OR ".join(f'"{t}"' for t in toks)
    rows = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                       "WHERE fts MATCH ? ORDER BY s LIMIT ?", (match, k)).fetchall()
    return [r[0] for r in rows]

# ---------- Chequeos puntuales (corregidos) ----------
sid = "168.1_T_CAMBIOR_D_0_0_26"
fila = meta[meta["serie_id"] == sid]
print("\nSerie BNA en el catálogo completo:", len(fila) > 0)
if len(fila):
    print(fila[["dataset_tema", "dataset_titulo", "serie_discontinuada", "serie_indice_final"]].T.to_string())
print("Serie BNA en el índice del proyecto:", (df["serie_id"] == sid).any())

soja = df[df["serie_titulo"].str.contains("soja", case=False, na=False)]
print("\nSeries con 'soja' en el título:", len(soja))
print(soja.groupby("dataset_titulo").size().sort_values(ascending=False).head(8).to_string())

# ---------- Series objetivo: test, usables, máx. 4 por dataset, hasta 100 ----------
test = df[(df["split"] == "test") & df["estado"].isin(["normal", "api_adelantada"])]
obj = (test.sample(frac=1, random_state=1).groupby("dataset_id").head(4)
           .head(100).reset_index(drop=True))
print("\nSeries objetivo:", len(obj), "| datasets:", obj["dataset_id"].nunique(),
      "| familias:", obj["familia"].nunique())

def armar_prompt(row):
    return f"""Sos un usuario argentino que busca un dato estadístico y no conoce los nombres técnicos de la base de datos.
Escribí UNA pregunta en español, natural, que pida el dato descrito abajo.
Reglas:
- No copies el título técnico literalmente ni uses guiones bajos.
- Si hay series parecidas, incluí lo necesario para distinguirla (frecuencia, base de precios u otro detalle del contexto).
- No incluyas fechas ni números.
Devolvé solo JSON: {{"pregunta": "..."}}

Título técnico: {row['serie_titulo']}
Distribución: {row['distribucion_titulo']}
Descripción: {row['serie_descripcion']}
Unidades: {row['serie_unidades']}
Frecuencia: {row['frecuencia']}
Dataset: {row['dataset_titulo']}"""

def llamar(row):
    msg = client.messages.create(model=MODEL, max_tokens=400,
                                 messages=[{"role": "user", "content": armar_prompt(row)}])
    txt = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text").strip()
    txt = txt[txt.find("{"): txt.rfind("}") + 1]
    return json.loads(txt)["pregunta"], msg.usage.input_tokens, msg.usage.output_tokens

# Prueba de UNA llamada: si falla, se muestra el error y se frena
filas = obj.to_dict("records")
try:
    p, ti, to = llamar(filas[0])
    print("\nLlamada de prueba OK:", p)
except Exception as e:
    print("\nLa llamada de prueba FALLÓ:", type(e).__name__, "-", str(e)[:400])
    raise

errores = Counter()
def generar(row):
    try:
        return llamar(row)
    except Exception as e:
        errores[f"{type(e).__name__}: {str(e)[:120]}"] += 1
        return None, 0, 0

with ThreadPoolExecutor(max_workers=2) as ex:
    res = list(ex.map(generar, filas))

obj["pregunta"] = [x[0] for x in res]
costo = sum(x[1] for x in res) * 0.10 / 1e6 + sum(x[2] for x in res) * 0.50 / 1e6
print(f"Preguntas generadas: {obj['pregunta'].notna().sum()}/{len(obj)} | costo ${costo:.4f}")
if errores:
    print("Errores:", dict(errores))
obj = obj[obj["pregunta"].notna()].copy()
assert len(obj) >= 60, "Se generaron muy pocas preguntas: no se calcula el recall"

# ---------- Recall ----------
rangos = []
for q, s in zip(obj["pregunta"], obj["serie_id"]):
    ids = buscar_ids(q, 10)
    rangos.append(ids.index(s) + 1 if s in ids else np.nan)
obj["rango"] = rangos
obj[["serie_id", "pregunta", "rango"]].to_csv(f"{RUTA}/eval_busqueda_preguntas.csv", index=False)

print()
for k in [1, 3, 5, 10]:
    print(f"recall@{k}: {(obj['rango'] <= k).mean():.1%}")
print("\nrecall@5 por frecuencia:")
print(obj.groupby("frecuencia")["rango"].agg(n="size", recall5=lambda s: (s <= 5).mean()).round(2))
print("\nrecall@5 por tema:")
print(obj.groupby("dataset_tema")["rango"].agg(n="size", recall5=lambda s: (s <= 5).mean()).round(2))

print("\nEjemplos de fallos (no aparece en los primeros 5):")
for _, r in obj[~(obj["rango"] <= 5)].head(8).iterrows():
    print("\nPregunta:", r["pregunta"])
    print("  Objetivo:", r["serie_titulo"], "|", str(r["distribucion_titulo"])[:70], "|", r["frecuencia"])
    for s in buscar_ids(r["pregunta"], 3):
        f = df[df["serie_id"] == s].iloc[0]
        print("   -", f["serie_titulo"], "|", str(f["distribucion_titulo"])[:70])

In [ ]:
!pip -q install anthropic
import os, re, sqlite3, shutil, unicodedata
import numpy as np
import pandas as pd
import anthropic
from google.colab import drive, userdata

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
meta = pd.read_csv(f"{RUTA}/catalogo_completo.csv", low_memory=False)
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=6)
MODEL = "claude-haiku-5-5"

# ---------- A) ¿Qué quedó afuera del recorte? ----------
TEMAS = ["Precios", "Sector Externo", "Dinero y Bancos", "Actividad",
         "Empleo e Ingresos", "Finanzas Públicas"]
t = meta["dataset_tema"].value_counts().rename("series").to_frame()
t["en_recorte"] = t.index.isin(TEMAS)
print("Temas del catálogo completo (top 20):")
print(t.head(20).to_string())

mf = meta[meta["dataset_tema"] == "Mercados Financieros"]
print("\nMercados Financieros:", len(mf), "series |", mf["dataset_titulo"].nunique(),
      "datasets | discontinuadas:", int((mf["serie_discontinuada"] == True).sum()))
print(mf["dataset_titulo"].value_counts().head(8).to_string())

fuera = meta[~meta["dataset_tema"].isin(TEMAS)]
cd = fuera[fuera["serie_titulo"].str.contains("cambio|dolar|usd", case=False, na=False)
           | fuera["dataset_titulo"].str.contains("cambio|dólar|dolar", case=False, na=False)]
print("\nSeries de cambio o dólar fuera del recorte:", len(cd))
print(cd.groupby(["dataset_tema", "dataset_titulo"]).size()
        .sort_values(ascending=False).head(8).to_string())

# ---------- B) ¿Por qué fallaron 31 generaciones? ----------
def armar_prompt(row):
    return f"""Sos un usuario argentino que busca un dato estadístico y no conoce los nombres técnicos de la base de datos.
Escribí UNA pregunta en español, natural, que pida el dato descrito abajo.
Reglas:
- No copies el título técnico literalmente ni uses guiones bajos.
- Si hay series parecidas, incluí lo necesario para distinguirla (frecuencia, base de precios u otro detalle del contexto).
- No incluyas fechas ni números.
Devolvé solo JSON: {{"pregunta": "..."}}

Título técnico: {row['serie_titulo']}
Distribución: {row['distribucion_titulo']}
Descripción: {row['serie_descripcion']}
Unidades: {row['serie_unidades']}
Frecuencia: {row['frecuencia']}
Dataset: {row['dataset_titulo']}"""

ya = pd.read_csv(f"{RUTA}/eval_busqueda_preguntas.csv")
test = df[(df["split"] == "test") & df["estado"].isin(["normal", "api_adelantada"])]
obj = (test.sample(frac=1, random_state=1).groupby("dataset_id").head(4)
           .head(100).reset_index(drop=True))
fallidas = obj[~obj["serie_id"].isin(ya["serie_id"])]
print(f"\nPreguntas guardadas: {len(ya)} | sin generar: {len(fallidas)}")
for _, row in fallidas.head(3).iterrows():
    msg = client.messages.create(model=MODEL, max_tokens=400,
                                 messages=[{"role": "user", "content": armar_prompt(row)}])
    texto = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
    det = getattr(msg.usage, "output_tokens_details", None)
    print("\n", row["serie_titulo"], "| stop_reason:", msg.stop_reason,
          "| tokens de salida:", msg.usage.output_tokens,
          "| tokens de razonamiento:", getattr(det, "thinking_tokens", None),
          "| bloques:", [b.type for b in msg.content])
    print("   texto:", repr(texto[:300]))

# ---------- C) Buscador actual vs. variantes (mismas preguntas, sin costo) ----------
DB = "/content/busqueda.db"
if not os.path.exists(DB):
    shutil.copy(f"{RUTA}/busqueda.db", DB)
con = sqlite3.connect(DB)

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}
EXTRA = {"podes", "pasar", "pasame", "serie", "historica", "historico", "evolucion",
         "dato", "datos", "medida", "medido", "medidos", "tal", "como", "viene",
         "publicada", "interesan", "contar", "no", "sin", "me"}

def buscar(q, k=10, prefijos=False, extra=False):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    if extra:
        toks = [t for t in toks if t not in EXTRA]
    toks = list(dict.fromkeys(toks))
    if not toks:
        return []
    partes = [f'"{t[:5]}"*' if (prefijos and len(t) >= 6) else f'"{t}"' for t in toks]
    rows = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                       "WHERE fts MATCH ? ORDER BY s LIMIT ?",
                       (" OR ".join(partes), k)).fetchall()
    return [r[0] for r in rows]

variantes = {"v1 actual": {},
             "solo prefijos": {"prefijos": True},
             "prefijos + genéricas": {"prefijos": True, "extra": True}}
rangos = {}
for nombre, kw in variantes.items():
    r = []
    for q, s in zip(ya["pregunta"], ya["serie_id"]):
        ids = buscar(q, 10, **kw)
        r.append(ids.index(s) + 1 if s in ids else np.nan)
    rangos[nombre] = np.array(r, dtype=float)

print(f"\nRecall (%) sobre las {len(ya)} preguntas guardadas:")
tabla = pd.DataFrame({n: {f"@{k}": (r <= k).mean() * 100 for k in [1, 3, 5, 10]}
                      for n, r in rangos.items()}).round(1)
print(tabla.to_string())
base, nuevo = rangos["v1 actual"], rangos["prefijos + genéricas"]
print("\nPreguntas que entran / salen del top 5 (v1 -> última variante):",
      int(((nuevo <= 5) & ~(base <= 5)).sum()), "/", int(((base <= 5) & ~(nuevo <= 5)).sum()))

In [ ]:
!pip -q install anthropic
import os, re, json, sqlite3, shutil, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
import anthropic
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive, userdata

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=6)
MODEL = "claude-haiku-5-5"

def armar_prompt(row):
    return f"""Sos un usuario argentino que busca un dato estadístico y no conoce los nombres técnicos de la base de datos.
Escribí UNA pregunta en español, natural, que pida el dato descrito abajo.
Reglas:
- No copies el título técnico literalmente ni uses guiones bajos.
- Si hay series parecidas, incluí lo necesario para distinguirla (frecuencia, base de precios u otro detalle del contexto).
- No incluyas fechas ni números.
Devolvé solo JSON: {{"pregunta": "..."}}

Título técnico: {row['serie_titulo']}
Distribución: {row['distribucion_titulo']}
Descripción: {row['serie_descripcion']}
Unidades: {row['serie_unidades']}
Frecuencia: {row['frecuencia']}
Dataset: {row['dataset_titulo']}"""

def llamar(row, max_tokens=2000):
    msg = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                 messages=[{"role": "user", "content": armar_prompt(row)}])
    if msg.stop_reason == "max_tokens":
        raise RuntimeError("max_tokens")
    txt = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
    txt = txt[txt.find("{"): txt.rfind("}") + 1]
    return json.loads(txt)["pregunta"], msg.usage.input_tokens, msg.usage.output_tokens

errores = Counter()
def generar(row):
    for mt in (2000, 4000):                       # un reintento con el doble de margen
        try:
            return llamar(row, mt)
        except RuntimeError:
            continue
        except Exception as e:
            errores[f"{type(e).__name__}: {str(e)[:100]}"] += 1
            return None, 0, 0
    errores["max_tokens incluso con 4000"] += 1
    return None, 0, 0

# ---------- Mismas series objetivo que antes; solo se generan las que faltan ----------
test = df[(df["split"] == "test") & df["estado"].isin(["normal", "api_adelantada"])]
obj = (test.sample(frac=1, random_state=1).groupby("dataset_id").head(4)
           .head(100).reset_index(drop=True))
ya = pd.read_csv(f"{RUTA}/eval_busqueda_preguntas.csv")
faltan = obj[~obj["serie_id"].isin(ya["serie_id"])]
print(f"Series objetivo: {len(obj)} | ya con pregunta: {len(ya)} | a generar: {len(faltan)}")

with ThreadPoolExecutor(max_workers=2) as ex:
    res = list(ex.map(generar, faltan.to_dict("records")))
costo = sum(x[1] for x in res) * 0.10 / 1e6 + sum(x[2] for x in res) * 0.50 / 1e6
print(f"Generadas: {sum(x[0] is not None for x in res)}/{len(faltan)} | costo ${costo:.4f}")
if errores:
    print("Errores:", dict(errores))

nuevas = faltan[["serie_id"]].copy()
nuevas["pregunta"] = [x[0] for x in res]
nuevas = nuevas[nuevas["pregunta"].notna()]
todas = pd.concat([ya[["serie_id", "pregunta"]].assign(origen="primera_tanda"),
                   nuevas.assign(origen="reintento_max_tokens")], ignore_index=True)
todas = todas.merge(df.drop_duplicates("serie_id")[["serie_id", "dataset_tema", "frecuencia"]],
                    on="serie_id", how="left")
print("Preguntas totales:", len(todas))

# ---------- Buscador (las tres variantes) ----------
DB = "/content/busqueda.db"
if not os.path.exists(DB):
    shutil.copy(f"{RUTA}/busqueda.db", DB)
con = sqlite3.connect(DB)

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}
EXTRA = {"podes", "pasar", "pasame", "serie", "historica", "historico", "evolucion",
         "dato", "datos", "medida", "medido", "medidos", "tal", "como", "viene",
         "publicada", "interesan", "contar", "no", "sin", "me"}

def buscar(q, k=10, prefijos=False, extra=False):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    if extra:
        toks = [t for t in toks if t not in EXTRA]
    toks = list(dict.fromkeys(toks))
    if not toks:
        return []
    partes = [f'"{t[:5]}"*' if (prefijos and len(t) >= 6) else f'"{t}"' for t in toks]
    rows = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                       "WHERE fts MATCH ? ORDER BY s LIMIT ?",
                       (" OR ".join(partes), k)).fetchall()
    return [r[0] for r in rows]

variantes = {"v1 actual": {}, "solo prefijos": {"prefijos": True},
             "prefijos + genéricas": {"prefijos": True, "extra": True}}
for nombre, kw in variantes.items():
    r = []
    for q, s in zip(todas["pregunta"], todas["serie_id"]):
        ids = buscar(q, 10, **kw)
        r.append(ids.index(s) + 1 if s in ids else np.nan)
    todas["rango_" + nombre] = r

print(f"\nRecall (%) sobre {len(todas)} preguntas:")
tabla = pd.DataFrame({n: {f"@{k}": (todas["rango_" + n] <= k).mean() * 100 for k in [1, 3, 5, 10]}
                      for n in variantes}).round(1)
print(tabla.to_string())

print("\nRecall@5 por origen de la pregunta (v1 actual):")
print(todas.groupby("origen")["rango_v1 actual"].agg(n="size", recall5=lambda s: (s <= 5).mean()).round(2))

todas.to_csv(f"{RUTA}/eval_busqueda_preguntas_v2.csv", index=False)
print("\nGuardado: eval_busqueda_preguntas_v2.csv")

In [ ]:
import os, re, shutil, sqlite3, time, unicodedata
import numpy as np
import pandas as pd
import requests
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
SAVE = True
BASE = "https://apis.datos.gob.ar/series/api/series"
HEADERS = {"User-Agent": "Mozilla/5.0"}
DATASET_TC = "Tipo de Cambio $-USD - Futuro Dólar"

df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
df["dataset_id"] = df["dataset_id"].astype(str)
meta = pd.read_csv(f"{RUTA}/catalogo_completo.csv", low_memory=False)
meta["dataset_id"] = meta["dataset_id"].astype(str)
assert {"familia", "txt_titulo", "txt_contexto", "split"} <= set(df.columns), \
    "catalogo_busqueda.csv no tiene la estructura esperada"

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

FREQ = {"R/P1M": "mensual", "R/P3M": "trimestral", "R/P1Y": "anual",
        "R/P6M": "semestral", "R/P1D": "diaria"}

# ---------- 1) Series del dataset de tipo de cambio ----------
tc = meta[(meta["dataset_tema"] == "Mercados Financieros")
          & (meta["dataset_titulo"] == DATASET_TC)].copy()
print("Series en el catálogo:", len(tc), "| discontinuadas:", int((tc["serie_discontinuada"] == True).sum()))
print("dataset_id:", tc["dataset_id"].unique().tolist())
vista = tc[["serie_id", "serie_titulo", "serie_descripcion", "serie_unidades",
            "indice_tiempo_frecuencia", "serie_indice_final", "serie_discontinuada"]].copy()
vista["serie_descripcion"] = vista["serie_descripcion"].astype(str).str[:50]
print(vista.to_string(index=False))

tc = tc[tc["serie_discontinuada"] != True].copy()
assert len(tc) > 0, "No quedaron series después de excluir discontinuadas"
assert not tc["serie_id"].isin(df["serie_id"]).any(), "Estas series ya están en el catálogo: no repetir"
assert not df["familia"].astype(str).isin(tc["dataset_id"].unique()).any(), \
    "La familia ya existe en la partición congelada"
print("\nSeries a incorporar:", len(tc))

# ---------- 2) Verificación contra la API ----------
def consultar(sid):
    status = None
    for intento in range(3):
        try:
            r = requests.get(BASE, params={"ids": sid, "limit": 1, "sort": "desc"},
                             headers=HEADERS, timeout=30)
            status = r.status_code
            if status == 200:
                js = r.json()
                data = js.get("data", [])
                return {"serie_id": sid, "status": 200, "n_total_api": js.get("count"),
                        "ultimo_api": data[0][0] if data else None,
                        "ultimo_valor": data[0][1] if data else None}
        except Exception:
            status = -1
        time.sleep(1 + intento)
    return {"serie_id": sid, "status": status, "n_total_api": None,
            "ultimo_api": None, "ultimo_valor": None}

filas = []
for sid in tc["serie_id"]:
    filas.append(consultar(sid))
    time.sleep(0.3)
ver = pd.DataFrame(filas)

def estado_de(n, desfase):
    if pd.isna(n) or n == 0:
        return "sin_datos"
    if pd.isna(desfase):
        return "sin_fecha"
    if desfase < -31:
        return "api_truncada"
    if desfase > 31:
        return "api_adelantada"
    return "normal"

tc = tc.merge(ver, on="serie_id", how="left")
tc["final"] = pd.to_datetime(tc["serie_indice_final"], errors="coerce")
tc["desfase_dias"] = (pd.to_datetime(tc["ultimo_api"], errors="coerce") - tc["final"]).dt.days
tc["estado"] = [estado_de(n, d) for n, d in zip(tc["n_total_api"], tc["desfase_dias"])]

print("\nStatus de la API:", ver["status"].value_counts().to_dict())
print("Estados:", tc["estado"].value_counts().to_dict())
print(tc[["serie_id", "serie_titulo", "n_total_api", "ultimo_api", "serie_indice_final",
          "desfase_dias", "estado"]].to_string(index=False))

# ---------- 3) Filas nuevas (familia propia, split train) ----------
nuevo = tc.copy()
nuevo["frecuencia"] = nuevo["indice_tiempo_frecuencia"].map(FREQ)
nuevo["txt_titulo"] = nuevo["serie_titulo"].map(norm)
nuevo["txt_contexto"] = (nuevo["distribucion_titulo"].map(norm) + " " + nuevo["serie_descripcion"].map(norm)
                         + " " + nuevo["dataset_titulo"].map(norm) + " " + nuevo["dataset_tema"].map(norm)
                         + " " + nuevo["serie_unidades"].map(norm) + " " + nuevo["frecuencia"].fillna(""))
nuevo["familia"] = nuevo["dataset_id"]
nuevo["split"] = "train"
nuevo["usable"] = nuevo["estado"].isin(["normal", "api_adelantada"])
nuevo = nuevo.reindex(columns=df.columns)
df2 = pd.concat([df, nuevo], ignore_index=True)
assert df2["serie_id"].is_unique
print("\nSeries totales:", len(df), "->", len(df2))
usables = df2[df2["estado"].isin(["normal", "api_adelantada"])]
print("Usables por split:", usables["split"].value_counts().to_dict(), "(antes: train 5696, val 417, test 408)")

# ---------- 4) Índices de búsqueda: antes y después ----------
def construir_indice(d, ruta):
    if os.path.exists(ruta):
        os.remove(ruta)
    con = sqlite3.connect(ruta)
    con.execute("""CREATE VIRTUAL TABLE fts USING fts5(
        serie_id UNINDEXED, titulo, contexto, tokenize='unicode61 remove_diacritics 2')""")
    con.executemany("INSERT INTO fts(serie_id, titulo, contexto) VALUES (?,?,?)",
                    d[["serie_id", "txt_titulo", "txt_contexto"]].fillna("").itertuples(index=False, name=None))
    con.commit()
    return con

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}

def buscar(con, q, k=10):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    toks = list(dict.fromkeys(toks))
    if not toks:
        return []
    partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]   # variante "solo prefijos"
    rows = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                       "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
    return [r[0] for r in rows]

con_antes = construir_indice(df, "/content/idx_antes.db")
con_desp = construir_indice(df2, "/content/idx_despues.db")

# ---------- 5) Recall antes y después (95 preguntas ya generadas) ----------
Q = pd.read_csv(f"{RUTA}/preguntas_{split}_v1.csv")
def recall(con):
    r = []
    for pregunta, s in zip(q["pregunta"], q["serie_id"]):
        ids = buscar(con, pregunta, 10)
        r.append(ids.index(s) + 1 if s in ids else np.nan)
    r = np.array(r, dtype=float)
    return {f"@{k}": round(float((r <= k).mean() * 100), 1) for k in [1, 3, 5, 10]}
print("\nRecall (variante prefijos) sobre", len(q), "preguntas:")
print(pd.DataFrame({"antes": recall(con_antes), "después": recall(con_desp)}).T.to_string())

# ---------- 6) ¿Aparece ahora el tipo de cambio? ----------
idx = df2.set_index("serie_id")
for consulta in ["tipo de cambio", "tipo de cambio dólar vendedor", "dólar", "cotización del dólar"]:
    print(f"\n>>> {consulta}")
    for pos, s in enumerate(buscar(con_desp, consulta, 5), 1):
        f = idx.loc[s]
        print(f"  {pos}. {s} | {f['serie_titulo']} | {str(f['distribucion_titulo'])[:50]} | {f['estado']}")
print("\nSerie BNA en el índice:", "168.1_T_CAMBIOR_D_0_0_26" in set(df2["serie_id"]))

# ---------- 7) Guardar ----------
if SAVE:
    shutil.copy(f"{RUTA}/catalogo_busqueda.csv", f"{RUTA}/catalogo_busqueda_antes_tc.csv")
    if os.path.exists(f"{RUTA}/busqueda.db"):
        shutil.copy(f"{RUTA}/busqueda.db", f"{RUTA}/busqueda_antes_tc.db")
    df2.to_csv(f"{RUTA}/catalogo_busqueda.csv", index=False)
    con_desp.close()
    shutil.copy("/content/idx_despues.db", f"{RUTA}/busqueda.db")
    tc.to_csv(f"{RUTA}/verificacion_tipo_cambio.csv", index=False)
    print("\nGuardado. Respaldos: catalogo_busqueda_antes_tc.csv y busqueda_antes_tc.db")
else:
    print("\nSAVE = False: no se guardó nada")

In [ ]:
import pandas as pd, sqlite3
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
d = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
u = d[d["estado"].isin(["normal", "api_adelantada"])]
print("Series:", len(d), "| usables por split:", u["split"].value_counts().to_dict())
con = sqlite3.connect(f"{RUTA}/busqueda.db")
print("Filas en el índice:", con.execute("select count(*) from fts").fetchone()[0])

In [ ]:
!pip -q install -U anthropic
import os, re, json, sqlite3, shutil, time, unicodedata
import numpy as np
import pandas as pd
import anthropic
from google.colab import drive, userdata

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
PARCIAL = f"{RUTA}/prueba_razonamiento_parcial.csv"
df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=3, timeout=60.0)
MODEL = "claude-haiku-5-5"

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

# Las mismas 20 series de TRAIN de antes (una por familia)
tr = df[(df["split"] == "train") & df["estado"].isin(["normal", "api_adelantada"])]
obj = tr.sample(frac=1, random_state=5).groupby("familia").head(1).head(20).reset_index(drop=True)
filas = obj.to_dict("records")
print("Series de la prueba:", len(filas))

def armar_prompt(row):
    return f"""Sos un usuario argentino que busca un dato estadístico y no conoce los nombres técnicos de la base de datos.
Escribí UNA pregunta en español, natural, que pida el dato descrito abajo.
Reglas:
- No copies el título técnico literalmente ni uses guiones bajos.
- Si hay series parecidas, incluí lo necesario para distinguirla (frecuencia, base de precios u otro detalle del contexto).
- No incluyas fechas ni números.
Devolvé solo JSON: {{"pregunta": "..."}}

Título técnico: {row['serie_titulo']}
Distribución: {row['distribucion_titulo']}
Descripción: {row['serie_descripcion']}
Unidades: {row['serie_unidades']}
Frecuencia: {row['frecuencia']}
Dataset: {row['dataset_titulo']}"""

def llamar(row, razona, max_tokens):
    kw = {} if razona else {"extra_body": {"thinking": {"type": "disabled"}}}
    msg = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                 messages=[{"role": "user", "content": armar_prompt(row)}], **kw)
    if msg.stop_reason == "max_tokens":
        raise RuntimeError("max_tokens")
    txt = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
    txt = txt[txt.find("{"): txt.rfind("}") + 1]
    det = getattr(msg.usage, "output_tokens_details", None)
    return {"pregunta": json.loads(txt)["pregunta"], "ent": msg.usage.input_tokens,
            "sal": msg.usage.output_tokens, "pens": getattr(det, "thinking_tokens", 0) or 0}

# Retomar lo ya hecho
reg = pd.read_csv(PARCIAL).to_dict("records") if os.path.exists(PARCIAL) else []
hechos = {(r["serie_id"], r["modo"]) for r in reg}
print("Ya hechos:", len(hechos))

t0 = time.time()
for i, row in enumerate(filas, 1):
    for razona in (False, True):
        modo = "con razonamiento" if razona else "sin razonamiento"
        if (row["serie_id"], modo) in hechos:
            continue
        x, err = None, ""
        for mt in (2000, 4000):
            try:
                x = llamar(row, razona, mt)
                break
            except RuntimeError:
                err = "max_tokens"
            except Exception as e:
                err = f"{type(e).__name__}: {str(e)[:100]}"
                break
        reg.append({"serie_id": row["serie_id"], "titulo": row["serie_titulo"], "modo": modo,
                    "ok": x is not None, "error": "" if x else err, **(x or {})})
        print(f"[{i}/20] {modo:17s} {'ok' if x else 'FALLÓ ' + err} | {time.time() - t0:.0f}s", flush=True)
    pd.DataFrame(reg).to_csv(PARCIAL, index=False)

R = pd.DataFrame(reg)
print("\nFallos:", int((~R["ok"]).sum()), "de", len(R))

# ---------- Filtros y recall ----------
DB = "/content/busqueda.db"
if not os.path.exists(DB):
    shutil.copy(f"{RUTA}/busqueda.db", DB)
con = sqlite3.connect(DB)
STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}

def buscar(q, k=10):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    toks = list(dict.fromkeys(toks))
    if not toks:
        return []
    partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
    rows = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                       "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
    return [x[0] for x in rows]

ok = R[R["ok"]].copy()
ok["palabras"] = ok["pregunta"].str.split().str.len()
ok["digitos"] = ok["pregunta"].str.contains(r"\d")
ok["copia_titulo"] = [len(norm(t).split()) >= 2 and norm(t) in norm(q)
                      for t, q in zip(ok["titulo"], ok["pregunta"])]
rangos = []
for q, s in zip(ok["pregunta"], ok["serie_id"]):
    ids = buscar(q, 10)
    rangos.append(ids.index(s) + 1 if s in ids else np.nan)
ok["rango"] = rangos
ok["costo"] = ok["ent"] * 0.10 / 1e6 + ok["sal"] * 0.50 / 1e6

print("\nComparación (mismas 20 series en ambos modos):")
resumen = ok.groupby("modo").agg(
    preguntas=("ok", "size"), sal_prom=("sal", "mean"), razon_prom=("pens", "mean"),
    costo_total=("costo", "sum"), costo_por_pregunta=("costo", "mean"),
    palabras_prom=("palabras", "mean"), pct_digitos=("digitos", "mean"),
    pct_copia_titulo=("copia_titulo", "mean"),
    recall5=("rango", lambda s: (s <= 5).mean()), recall10=("rango", lambda s: (s <= 10).mean()))
print(resumen.round(5).T.to_string())

print("\nEjemplos lado a lado:")
pv = ok.pivot(index="titulo", columns="modo", values="pregunta").dropna()
for t, f in pv.head(6).iterrows():
    print("\nSerie:", t)
    print("  SIN :", f["sin razonamiento"])
    print("  CON :", f["con razonamiento"])

ok.to_csv(f"{RUTA}/prueba_razonamiento.csv", index=False)
print("\nGuardado: prueba_razonamiento.csv")

In [ ]:
!pip -q install -U anthropic
import os, re, json, sqlite3, shutil, time, unicodedata
import numpy as np
import pandas as pd
import anthropic
from google.colab import drive, userdata

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
PARCIAL = f"{RUTA}/eval_busqueda_val_parcial.csv"
df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=3, timeout=60.0)
MODEL = "claude-haiku-5-5"

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

# ---------- Series objetivo: VALIDACIÓN, usables, máx. 6 por dataset, hasta 100 ----------
val = df[(df["split"] == "val") & df["estado"].isin(["normal", "api_adelantada"])]
obj = (val.sample(frac=1, random_state=11).groupby("dataset_id").head(6)
          .head(100).reset_index(drop=True))
filas = obj.to_dict("records")
print("Series objetivo:", len(filas), "| datasets:", obj["dataset_id"].nunique(),
      "| familias:", obj["familia"].nunique())

def armar_prompt(row):
    return f"""Sos un usuario argentino que busca un dato estadístico y no conoce los nombres técnicos de la base de datos.
Escribí UNA pregunta en español, natural, que pida el dato descrito abajo.
Reglas:
- No copies el título técnico literalmente ni uses guiones bajos.
- Si hay series parecidas, incluí lo necesario para distinguirla (frecuencia, base de precios u otro detalle del contexto).
- No incluyas fechas ni números.
Devolvé solo JSON: {{"pregunta": "..."}}

Título técnico: {row['serie_titulo']}
Distribución: {row['distribucion_titulo']}
Descripción: {row['serie_descripcion']}
Unidades: {row['serie_unidades']}
Frecuencia: {row['frecuencia']}
Dataset: {row['dataset_titulo']}"""

def llamar(row, max_tokens):
    msg = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                 messages=[{"role": "user", "content": armar_prompt(row)}])
    if msg.stop_reason == "max_tokens":
        raise RuntimeError("max_tokens")
    txt = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
    txt = txt[txt.find("{"): txt.rfind("}") + 1]
    return {"pregunta": json.loads(txt)["pregunta"], "ent": msg.usage.input_tokens,
            "sal": msg.usage.output_tokens}

# Retomar: se conservan solo las filas que salieron bien; las fallidas se reintentan
reg = pd.read_csv(PARCIAL).to_dict("records") if os.path.exists(PARCIAL) else []
reg = [r for r in reg if r["ok"]]
hechos = {r["serie_id"] for r in reg}
print("Ya hechas:", len(hechos))

t0 = time.time()
for i, row in enumerate(filas, 1):
    if row["serie_id"] in hechos:
        continue
    x, err = None, ""
    for mt in (2000, 4000):
        try:
            x = llamar(row, mt)
            break
        except RuntimeError:
            err = "max_tokens"
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:100]}"
            break
    reg.append({"serie_id": row["serie_id"], "ok": x is not None, "error": "" if x else err, **(x or {})})
    if i % 10 == 0 or not x:
        print(f"[{i}/{len(filas)}] {'ok' if x else 'FALLÓ ' + err} | {time.time() - t0:.0f}s", flush=True)
    pd.DataFrame(reg).to_csv(PARCIAL, index=False)

R = pd.DataFrame(reg)
print(f"\nGeneradas: {int(R['ok'].sum())}/{len(filas)} | fallos: {int((~R['ok']).sum())}")
assert R["ok"].sum() >= 60, "Se generaron muy pocas preguntas: no se calcula el recall"

# ---------- Buscador (variante "solo prefijos", la elegida) ----------
DB = "/content/busqueda.db"
if not os.path.exists(DB):
    shutil.copy(f"{RUTA}/busqueda.db", DB)
con = sqlite3.connect(DB)
STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}

def buscar(q, k=10):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    toks = list(dict.fromkeys(toks))
    if not toks:
        return []
    partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
    rows = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                       "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
    return [x[0] for x in rows]

ok = R[R["ok"]].merge(df.drop_duplicates("serie_id")[
        ["serie_id", "serie_titulo", "distribucion_titulo", "frecuencia", "dataset_tema", "familia"]],
        on="serie_id", how="left")
rangos = []
for q, s in zip(ok["pregunta"], ok["serie_id"]):
    ids = buscar(q, 10)
    rangos.append(ids.index(s) + 1 if s in ids else np.nan)
ok["rango"] = rangos
ok["palabras"] = ok["pregunta"].str.split().str.len()

def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return c - h, c + h

n = len(ok)
print(f"\nRECALL EN VALIDACIÓN (n = {n}, variante prefijos, una sola corrida):")
for k in [1, 3, 5, 10]:
    hits = int((ok["rango"] <= k).sum())
    lo, hi = wilson(hits, n)
    print(f"  recall@{k}: {hits / n:.1%}  (IC 95%: {lo:.0%} a {hi:.0%})")
print(f"Palabras por pregunta (prom.): {ok['palabras'].mean():.1f}")
costo = R["ent"].sum() * 0.10 / 1e6 + R["sal"].sum() * 0.50 / 1e6
print(f"Costo: ${costo:.4f}")

print("\nrecall@5 por frecuencia y por tema (n chicos: solo orientativo):")
print(ok.groupby("frecuencia")["rango"].agg(n="size", recall5=lambda s: (s <= 5).mean()).round(2).to_string())
print(ok.groupby("dataset_tema")["rango"].agg(n="size", recall5=lambda s: (s <= 5).mean()).round(2).to_string())

print("\nFallos (solo para documentar el tipo de error; NO para ajustar el buscador):")
for _, r in ok[~(ok["rango"] <= 5)].head(6).iterrows():
    print("\nPregunta:", r["pregunta"])
    print("  Objetivo:", r["serie_titulo"], "|", str(r["distribucion_titulo"])[:60])

ok.to_csv(f"{RUTA}/eval_busqueda_val.csv", index=False)
print("\nGuardado: eval_busqueda_val.csv")

In [ ]:
import anthropic
from google.colab import userdata

c = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
m = c.messages.create(model="claude-haiku-5-5", max_tokens=50,
                      messages=[{"role": "user", "content": "Decí OK"}],
                      extra_body={"thinking": {"type": "disabled"}})
print(m.content[0].text)

In [ ]:
!pip -q install -U anthropic
import os, re, json, sqlite3, shutil, time, unicodedata
import numpy as np
import pandas as pd
import anthropic
from google.colab import drive, userdata

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
PARCIAL = f"{RUTA}/eval_busqueda_val_cortas_parcial.csv"
df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
df["dataset_id"] = df["dataset_id"].astype(str)
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=3, timeout=60.0)
MODEL = "claude-haiku-5-5"

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

# ---------- Series objetivo: validación, usables, DISTINTAS de las 94 ya usadas ----------
usadas = set(pd.read_csv(f"{RUTA}/eval_busqueda_val.csv")["serie_id"])
val = df[(df["split"] == "val") & df["estado"].isin(["normal", "api_adelantada"])
         & ~df["serie_id"].isin(usadas)]
obj = (val.sample(frac=1, random_state=21).groupby("dataset_id").head(6)
          .head(100).reset_index(drop=True))
filas = obj.to_dict("records")
print("Series objetivo:", len(filas), "| datasets:", obj["dataset_id"].nunique(),
      "| familias:", obj["familia"].nunique(), "| ya usadas excluidas:", len(usadas))

def armar_prompt(row):
    return f"""Escribí cómo preguntaría esto una persona común de Argentina que NO conoce la base de datos y escribe rápido, por ejemplo en un chat.
Reglas:
- Entre 4 y 10 palabras. Tono coloquial, sin formalidades.
- Mencioná solo el tema principal. No des detalles técnicos (base, metodología, unidad, desagregación).
- No uses números, años ni fechas.
- No copies el título técnico.
Devolvé solo JSON: {{"pregunta": "..."}}

Dato que busca:
Título técnico: {row['serie_titulo']}
Descripción: {row['serie_descripcion']}
Dataset: {row['dataset_titulo']}
Frecuencia: {row['frecuencia']}"""

TOT = {"ent": 0, "sal": 0}     # tokens gastados en esta sesión (incluye reintentos)

def llamar(row, max_tokens):
    msg = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                 messages=[{"role": "user", "content": armar_prompt(row)}])
    TOT["ent"] += msg.usage.input_tokens
    TOT["sal"] += msg.usage.output_tokens
    if msg.stop_reason == "max_tokens":
        raise RuntimeError("max_tokens")
    txt = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
    txt = txt[txt.find("{"): txt.rfind("}") + 1]
    return {"pregunta": json.loads(txt)["pregunta"]}

def generar(row):
    err = ""
    for _ in range(3):
        try:
            x = llamar(row, 3000)
        except RuntimeError:
            err = "max_tokens"
            continue
        except Exception as e:
            return None, f"{type(e).__name__}: {str(e)[:100]}"
        q = x["pregunta"]
        if re.search(r"\d", q) or not (3 <= len(q.split()) <= 14):
            err = "filtro (dígitos o largo)"
            continue
        return x, ""
    return None, err

# Retomar: se conservan solo las filas que salieron bien
reg = pd.read_csv(PARCIAL).to_dict("records") if os.path.exists(PARCIAL) else []
reg = [r for r in reg if r["ok"]]
hechos = {r["serie_id"] for r in reg}
print("Ya hechas:", len(hechos))

t0 = time.time()
for i, row in enumerate(filas, 1):
    if row["serie_id"] in hechos:
        continue
    x, err = generar(row)
    reg.append({"serie_id": row["serie_id"], "ok": x is not None, "error": err,
                "pregunta": x["pregunta"] if x else ""})
    if i % 10 == 0 or not x:
        print(f"[{i}/{len(filas)}] {'ok' if x else 'FALLÓ ' + err} | {time.time() - t0:.0f}s", flush=True)
    pd.DataFrame(reg).to_csv(PARCIAL, index=False)

R = pd.DataFrame(reg)
print(f"\nGeneradas: {int(R['ok'].sum())}/{len(filas)} | fallos: {int((~R['ok']).sum())}")
assert R["ok"].sum() >= 60, "Se generaron muy pocas preguntas: no se calcula el recall"
print(f"Costo de esta sesión: ${TOT['ent'] * 0.10 / 1e6 + TOT['sal'] * 0.50 / 1e6:.4f}")

# ---------- Buscador (variante de prefijos, la elegida) ----------
DB = "/content/busqueda.db"
if not os.path.exists(DB):
    shutil.copy(f"{RUTA}/busqueda.db", DB)
con = sqlite3.connect(DB)
STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}

def buscar(q, k=10):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    toks = list(dict.fromkeys(toks))
    if not toks:
        return []
    partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
    rows = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                       "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
    return [x[0] for x in rows]

info = df.drop_duplicates("serie_id").set_index("serie_id")
ok = R[R["ok"]].merge(
    df.drop_duplicates("serie_id")[["serie_id", "serie_titulo", "distribucion_titulo",
                                    "dataset_id", "frecuencia", "dataset_tema"]],
    on="serie_id", how="left")
rango, rango_ds = [], []
for q, s, d in zip(ok["pregunta"], ok["serie_id"], ok["dataset_id"]):
    ids = buscar(q, 10)
    rango.append(ids.index(s) + 1 if s in ids else np.nan)
    ds = [info.loc[i, "dataset_id"] for i in ids]
    rango_ds.append(ds.index(d) + 1 if d in ds else np.nan)
ok["rango"] = rango
ok["rango_dataset"] = rango_ds
ok["palabras"] = ok["pregunta"].str.split().str.len()

def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return c - h, c + h

n = len(ok)
print(f"\nPREGUNTAS CORTAS EN VALIDACIÓN (n = {n}, una sola corrida, no se ajusta nada con esto)")
print(f"Palabras por pregunta (prom.): {ok['palabras'].mean():.1f}")
print("\nSerie exacta en los primeros k resultados:")
for k in [1, 3, 5, 10]:
    hits = int((ok["rango"] <= k).sum())
    lo, hi = wilson(hits, n)
    print(f"  recall@{k}: {hits / n:.1%}  (IC 95%: {lo:.0%} a {hi:.0%})")
print("\nAlgún resultado del mismo dataset en los primeros k:")
for k in [1, 3, 5, 10]:
    hits = int((ok["rango_dataset"] <= k).sum())
    lo, hi = wilson(hits, n)
    print(f"  dataset@{k}: {hits / n:.1%}  (IC 95%: {lo:.0%} a {hi:.0%})")
print("\nReferencia, preguntas largas (n = 94): recall@1 58,5% | @3 76,6% | @5 86,2% | @10 94,7%")

print("\nEjemplos de preguntas generadas:")
for _, r in ok.head(8).iterrows():
    print(f"  - {r['pregunta']}   [objetivo: {r['serie_titulo']}]")

print("\nFallos (solo para documentar; NO para ajustar el buscador):")
for _, r in ok[~(ok["rango"] <= 5)].head(6).iterrows():
    print("\nPregunta:", r["pregunta"])
    print("  Objetivo:", r["serie_titulo"], "|", str(r["distribucion_titulo"])[:60],
          "| rango:", r["rango"], "| rango dataset:", r["rango_dataset"])

ok.to_csv(f"{RUTA}/eval_busqueda_val_cortas.csv", index=False)
print("\nGuardado: eval_busqueda_val_cortas.csv")

In [ ]:
import os, re, sqlite3, shutil, unicodedata
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
df["dataset_id"] = df["dataset_id"].astype(str)
ok = pd.read_csv(f"{RUTA}/eval_busqueda_val_cortas.csv")
info = df.drop_duplicates("serie_id").set_index("serie_id")

DB = "/content/busqueda.db"
if not os.path.exists(DB):
    shutil.copy(f"{RUTA}/busqueda.db", DB)
con = sqlite3.connect(DB)

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}

def buscar(q, k=10):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    toks = list(dict.fromkeys(toks))
    if not toks:
        return [], toks
    partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
    rows = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                       "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
    return [r[0] for r in rows], toks

# 1) ¿Cuántas consultas devuelven 0 resultados?
n_res, n_tok = [], []
for q in ok["pregunta"]:
    ids, toks = buscar(q, 10)
    n_res.append(len(ids)); n_tok.append(len(toks))
ok["n_resultados"], ok["n_palabras_consulta"] = n_res, n_tok
print("Consultas con 0 resultados:", int((ok["n_resultados"] == 0).sum()), "de", len(ok))
print("Consultas con menos de 10 resultados:", int((ok["n_resultados"] < 10).sum()))
print("Palabras útiles por consulta (prom.):", round(ok["n_palabras_consulta"].mean(), 1))

# 2) Fallos por tipo: sin resultados / con resultados pero sin el dataset correcto
fallo = ok[~(ok["rango_dataset"] <= 10)]
print("\nSin el dataset correcto en el top 10:", len(fallo))
print("  de ellos, con 0 resultados:", int((fallo["n_resultados"] == 0).sum()))

# 3) Qué devuelve el buscador en 8 de esos fallos
for _, r in fallo.head(8).iterrows():
    ids, toks = buscar(r["pregunta"], 3)
    print(f"\nPregunta: {r['pregunta']}")
    print(f"  Palabras buscadas: {toks}")
    print(f"  Objetivo: {r['serie_titulo']} | {str(r['distribucion_titulo'])[:60]}")
    for s in ids:
        print("   -", info.loc[s, "serie_titulo"], "|", str(info.loc[s, "distribucion_titulo"])[:50])

In [ ]:
import os, json, time, shutil, sqlite3
import pandas as pd
import requests
from datetime import datetime, timezone
from google.colab import drive

print("CELDA: CACHÉ DE PEDIDOS + EXPLORACIÓN PARA PLANTILLAS")
drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
SAVE = False        # False: la caché queda solo en /content; True: se copia a Drive
BASE = "https://apis.datos.gob.ar/series/api/series"
HEADERS = {"User-Agent": "Mozilla/5.0"}
DB_DRIVE = f"{RUTA}/cache_api.db"
DB_LOCAL = "/content/cache_api.db"

# Si ya hay una caché en Drive, se parte de ella (no se pisa nada)
if os.path.exists(DB_DRIVE) and not os.path.exists(DB_LOCAL):
    shutil.copy(DB_DRIVE, DB_LOCAL)
    print("Caché existente copiada desde Drive")

class CacheAPI:
    PARAMS_OK = {"ids", "start_date", "end_date", "limit", "start", "sort",
                 "collapse", "collapse_aggregation", "representation_mode"}

    def __init__(self, ruta_db, red=True, pausa=0.3):
        self.con = sqlite3.connect(ruta_db)
        self.con.execute("CREATE TABLE IF NOT EXISTS cache (clave TEXT PRIMARY KEY, "
                         "status INTEGER, cuerpo TEXT, consultado TEXT)")
        self.con.commit()
        self.red, self.pausa = red, pausa
        self.stats = {"aciertos": 0, "fallos_de_cache": 0, "error_de_red": 0}

    @classmethod
    def clave(cls, params):
        extra = set(params) - cls.PARAMS_OK
        if extra:
            raise ValueError(f"Parámetros no previstos: {sorted(extra)}")
        p = {k: str(v) for k, v in params.items() if v is not None}
        return json.dumps(dict(sorted(p.items())), ensure_ascii=False)

    def get(self, params):
        clave = self.clave(params)
        fila = self.con.execute("SELECT status, cuerpo, consultado FROM cache WHERE clave=?",
                                (clave,)).fetchone()
        if fila:
            self.stats["aciertos"] += 1
            return {"status": fila[0], "json": json.loads(fila[1]),
                    "consultado": fila[2], "de_cache": True}
        self.stats["fallos_de_cache"] += 1
        if not self.red:
            return {"status": None, "json": None, "consultado": None,
                    "de_cache": False, "error": "no está en la caché"}
        for intento in range(3):
            try:
                r = requests.get(BASE, params=json.loads(clave), headers=HEADERS, timeout=30)
                time.sleep(self.pausa)
                if r.status_code in (200, 400):
                    js = r.json()
                    ahora = datetime.now(timezone.utc).isoformat(timespec="seconds")
                    self.con.execute("INSERT OR REPLACE INTO cache VALUES (?,?,?,?)",
                                     (clave, r.status_code, json.dumps(js, ensure_ascii=False), ahora))
                    self.con.commit()
                    return {"status": r.status_code, "json": js,
                            "consultado": ahora, "de_cache": False}
            except Exception:
                pass
            time.sleep(1 + intento)
        self.stats["error_de_red"] += 1
        return {"status": None, "json": None, "consultado": None,
                "de_cache": False, "error": "falló la red"}

    def tamano(self):
        return self.con.execute("SELECT COUNT(*) FROM cache").fetchone()[0]

c = CacheAPI(DB_LOCAL)

def datos(resp):
    return (resp["json"] or {}).get("data")

def error(resp):
    return f"status={resp['status']} | {str(resp['json'])[:200]}"

# ---------- Exploración: una serie "normal" de train por frecuencia ----------
df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
tr = df[(df["split"] == "train") & (df["estado"] == "normal")]
sid_rep = None

for freq in ["mensual", "trimestral", "anual", "diaria"]:
    cand = tr[tr["frecuencia"] == freq]
    if cand.empty:
        print(f"\n=== {freq}: no hay series candidatas")
        continue
    row = cand.sample(1, random_state=3).iloc[0]
    sid = row["serie_id"]
    print(f"\n=== {freq} | {sid} | {row['serie_titulo']}")

    r0 = c.get({"ids": sid, "limit": 1, "sort": "asc"})
    if r0["status"] != 200 or not datos(r0):
        print("  sin datos:", error(r0))
        continue
    sid_rep = sid
    n = r0["json"]["count"]
    print("  count:", n, "| primer dato:", datos(r0)[0])

    # 1) ¿'start' funciona como desplazamiento?
    a = datos(c.get({"ids": sid, "limit": 3, "sort": "asc", "start": 0}))
    b = datos(c.get({"ids": sid, "limit": 3, "sort": "asc", "start": 2}))
    print("  start=0:", a)
    print("  start=2:", b)
    print("  ¿start es un desplazamiento?", bool(a and b and len(a) >= 3 and b[0] == a[2]))

    # 2) Un punto del medio de la serie y su consulta por período
    medio = datos(c.get({"ids": sid, "limit": 1, "sort": "asc", "start": n // 2}))
    if not medio:
        print("  no se pudo leer el punto medio")
        continue
    fecha = medio[0][0]
    p = c.get({"ids": sid, "start_date": fecha, "end_date": fecha})
    print("  punto medio:", medio[0], "| consulta por período:", datos(p))

    # 3) Variación interanual de ese período
    v = c.get({"ids": sid, "start_date": fecha, "end_date": fecha,
               "representation_mode": "percent_change_a_year_ago"})
    print("  interanual:", datos(v) if v["status"] == 200 else error(v))

    # 4) Período fuera de rango
    f = c.get({"ids": sid, "start_date": "2090-01-01", "end_date": "2090-12-31"})
    print("  fuera de rango:", f["status"], datos(f))

# ---------- Prueba de la caché ----------
if sid_rep:
    rep = c.get({"ids": sid_rep, "limit": 1, "sort": "asc"})
    print("\nRepetición de una consulta; ¿viene de la caché?", rep["de_cache"])
print("Estadísticas:", c.stats, "| entradas en la caché:", c.tamano())

c.con.commit()
if SAVE:
    shutil.copy(DB_LOCAL, DB_DRIVE)
    print("Copiada a Drive:", DB_DRIVE)
else:
    print("SAVE = False: la caché queda solo en /content")

In [ ]:
print("CELDA: EXPLORACIÓN 2 (FORMATO DE FECHAS Y VARIACIÓN INTERANUAL)")
assert "c" in globals(), "Ejecutá antes la celda de la caché (define 'c')"
import pandas as pd

def punto(resp):
    d = datos(resp)
    return d[0] if d else None

for freq in ["mensual", "trimestral", "anual", "diaria"]:
    row = tr[tr["frecuencia"] == freq].sample(1, random_state=3).iloc[0]
    sid = row["serie_id"]
    print(f"\n=== {freq} | {sid} | {row['serie_titulo']}")

    r0 = c.get({"ids": sid, "limit": 1, "sort": "asc"})
    n = r0["json"]["count"]
    fecha = punto(c.get({"ids": sid, "limit": 1, "sort": "asc", "start": n // 2}))[0]
    previa = (pd.to_datetime(fecha) - pd.DateOffset(years=1)).strftime("%Y-%m-%d")
    print("  fecha:", fecha, "| un año antes:", previa)

    # A) Variación interanual con una ventana que incluye el año anterior
    v = c.get({"ids": sid, "start_date": previa, "end_date": fecha, "limit": 1000})
    d = dict(datos(v) or [])
    v_t, v_prev = d.get(fecha), d.get(previa)
    a = c.get({"ids": sid, "start_date": previa, "end_date": fecha, "limit": 1000,
               "representation_mode": "percent_change_a_year_ago"})
    filas_api = datos(a) if a["status"] == 200 else error(a)
    print("  valores crudos: t =", v_t, "| un año antes =", v_prev, "| filas en la ventana:", len(d))
    print("  API interanual (últimas filas):",
          filas_api if isinstance(filas_api, str) else filas_api[-3:])
    if v_t is not None and v_prev not in (None, 0):
        print("  cálculo propio (v_t / v_prev - 1):", round(v_t / v_prev - 1, 6))

    # B) Formato de fecha corto como inicio y fin
    corto = fecha[:4] if freq == "anual" else fecha[:7]
    pc = c.get({"ids": sid, "start_date": corto, "end_date": corto})
    f = datos(pc) or []
    print(f"  fecha corta '{corto}': status={pc['status']} | filas={len(f)} | primeras={f[:2]}")

    # C) Variación período a período con una ventana de un solo punto
    pcv = c.get({"ids": sid, "start_date": fecha, "end_date": fecha,
                 "representation_mode": "percent_change"})
    print("  percent_change con un solo punto:", datos(pcv) if pcv["status"] == 200 else error(pcv))

print("\nEstadísticas:", c.stats, "| entradas en la caché:", c.tamano())

In [ ]:
import os, json, time, shutil, random, zlib
from collections import Counter
import pandas as pd

print("CELDA: ESPECIFICACIONES DE PREGUNTAS (4 TIPOS) CON ETIQUETAS DESDE LA API")
assert all(x in globals() for x in ["c", "datos", "DB_LOCAL", "DB_DRIVE"]), \
    "Ejecutá antes la celda de la caché (define 'c', 'datos', 'DB_LOCAL' y 'DB_DRIVE')"

RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
SAVE = True         # seguro: solo crea archivos nuevos; la caché es de solo agregado
N_SERIES = 60       # series con datos (subir después de revisar el resultado)
N_SIN_DATOS = 10    # series sin datos para el tipo "sin_datos"
CAP_FAM = 40        # tope de series por familia
MAX_VAR = 5.0       # variaciones interanuales mayores a 500% se descartan

LAG = {"mensual": 12, "trimestral": 4, "anual": 1}   # diarias y semestrales: fuera de esta versión

def periodo(fecha, freq):
    return fecha[:4] if freq == "anual" else fecha[:7]

def desplazar(fecha, anios):
    return (pd.to_datetime(fecha) + pd.DateOffset(years=anios)).strftime("%Y-%m-%d")

def periodo_al_azar(rng, freq):
    anio = rng.randint(2015, 2024)
    if freq == "anual":
        return str(anio)
    mes = rng.choice([1, 4, 7, 10]) if freq == "trimestral" else rng.randint(1, 12)
    return f"{anio}-{mes:02d}"

def fmt(v):
    return f"{v:.4f}".rstrip("0").rstrip(".")

def nombre(row):
    return str(row["serie_titulo"]).replace("_", " ")

def unidad(row):
    u = row["serie_unidades"]
    return f" ({u})" if isinstance(u, str) and u.strip() else ""

# ---------- Selección de series (solo TRAIN) ----------
df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
base = df[(df["split"] == "train") & df["frecuencia"].isin(LAG)]
con_datos = base[base["estado"].isin(["normal", "api_adelantada"])]
sin_datos = base[base["estado"] == "sin_datos"]

def elegir(d, n):
    return d.sample(frac=1, random_state=7).groupby("familia").head(CAP_FAM).head(n)

S1, S0 = elegir(con_datos, N_SERIES), elegir(sin_datos, N_SIN_DATOS)
print(f"Series con datos: {len(S1)} (familias: {S1['familia'].nunique()}) | sin datos: {len(S0)}")

specs, descartes = [], Counter()

def registrar(tipo, row, params, resp, final, caso=""):
    specs.append({"tipo": tipo, "caso": caso, "serie_id": row["serie_id"], "familia": row["familia"],
                  "frecuencia": row["frecuencia"], "serie_titulo": row["serie_titulo"],
                  "serie_unidades": row["serie_unidades"],
                  "params": json.dumps(params, ensure_ascii=False, sort_keys=True),
                  "status": resp["status"], "data": json.dumps(datos(resp), ensure_ascii=False),
                  "consultado": resp["consultado"], "respuesta_final": final})

t0 = time.time()
for k, row in enumerate(S1.to_dict("records"), 1):
    sid, freq = row["serie_id"], row["frecuencia"]
    rng = random.Random(zlib.crc32(sid.encode()))      # semilla estable por serie
    nom, uni = nombre(row), unidad(row)

    r0 = c.get({"ids": sid, "limit": 1, "sort": "asc"})
    rl = c.get({"ids": sid, "limit": 1, "sort": "desc"})
    if not (r0["status"] == 200 and datos(r0)):
        descartes["la serie no devolvió datos al verificar"] += 1
        continue
    n = r0["json"]["count"]
    primera = datos(r0)[0][0]
    ultimo = datos(rl)[0] if (rl["status"] == 200 and datos(rl)) else None

    # --- ultimo_dato ---
    if ultimo and ultimo[1] is not None:
        registrar("ultimo_dato", row, {"ids": sid, "limit": 1, "sort": "desc"}, rl,
                  f"El último dato disponible de {nom}{uni} es {fmt(ultimo[1])}, de la fecha {ultimo[0]}.")
    else:
        descartes["ultimo_dato: respuesta vacía o nula"] += 1

    # --- valor_puntual ---
    pt = datos(c.get({"ids": sid, "limit": 1, "sort": "asc", "start": rng.randrange(n)}))
    if pt:
        p = periodo(pt[0][0], freq)
        params = {"ids": sid, "start_date": p, "end_date": p}
        rr = c.get(params)
        d = datos(rr)
        if rr["status"] == 200 and d and d[0][1] is not None:
            registrar("valor_puntual", row, params, rr, f"{nom}{uni}: {fmt(d[0][1])} en {p}.")
        else:
            descartes["valor_puntual: sin valor"] += 1

    # --- variacion_interanual (ventana que incluye el año previo) ---
    lag = LAG[freq]
    if n > lag:
        pt = datos(c.get({"ids": sid, "limit": 1, "sort": "asc", "start": rng.randrange(lag, n)}))
        if pt:
            d0 = pt[0][0]
            prev = desplazar(d0, -1)
            params = {"ids": sid, "start_date": prev, "end_date": d0,
                      "representation_mode": "percent_change_a_year_ago"}
            rr = c.get(params)
            d = datos(rr)
            if rr["status"] == 200 and d and d[-1][0] == d0 and d[-1][1] is not None:
                v = d[-1][1]
                if abs(v) > MAX_VAR:
                    descartes["variacion: valor absurdo (> 500%)"] += 1
                else:
                    registrar("variacion_interanual", row, params, rr,
                              f"La variación interanual de {nom} en {periodo(d0, freq)} fue de "
                              f"{v * 100:.2f}% (respecto de {periodo(prev, freq)}).")
            else:
                descartes["variacion: sin dato interanual"] += 1

    # --- sin_datos: período fuera del rango de la serie ---
    antes = (ultimo is None) or rng.random() < 0.5
    fuera = desplazar(primera, -3) if antes else desplazar(ultimo[0], 2)
    p = periodo(fuera, freq)
    params = {"ids": sid, "start_date": p, "end_date": p}
    rr = c.get(params)
    if rr["status"] == 200 and datos(rr) == []:
        registrar("sin_datos", row, params, rr, f"La API no devuelve datos de {nom} para el período {p}.",
                  caso="fuera_de_rango_antes" if antes else "fuera_de_rango_despues")
    else:
        descartes["sin_datos: la API devolvió datos o error"] += 1

    if k % 10 == 0:
        print(f"[{k}/{len(S1)}] {time.time() - t0:.0f}s | specs: {len(specs)}", flush=True)

# --- sin_datos: series que no devuelven nada ---
for row in S0.to_dict("records"):
    rng = random.Random(zlib.crc32(row["serie_id"].encode()))
    p = periodo_al_azar(rng, row["frecuencia"])
    params = {"ids": row["serie_id"], "start_date": p, "end_date": p}
    rr = c.get(params)
    if rr["status"] == 200 and datos(rr) == []:
        registrar("sin_datos", row, params, rr,
                  f"La API no devuelve datos para la serie {nombre(row)}.", caso="serie_sin_datos")
    else:
        descartes["serie_sin_datos: la API devolvió datos o error"] += 1

# ---------- Resumen ----------
E = pd.DataFrame(specs)
print("\nEspecificaciones:", len(E), "| series distintas:", E["serie_id"].nunique(),
      "| familias:", E["familia"].nunique())
print(E.groupby(["tipo", "caso"]).size().to_string())
print("Descartes:", dict(descartes))
print("Caché:", c.stats, "| entradas:", c.tamano(), f"| {time.time() - t0:.0f}s")

v3 = E[E["tipo"] == "variacion_interanual"]["data"].map(lambda s: json.loads(s)[-1][1])
if len(v3):
    print("\nVariación interanual (fracción):")
    print(v3.describe().round(3).to_string())

for tipo in E["tipo"].unique():
    print(f"\n--- {tipo}")
    for _, r in E[E["tipo"] == tipo].head(2).iterrows():
        print("  serie:", r["serie_titulo"], "|", r["frecuencia"])
        print("  consulta:", r["params"])
        print("  respuesta final:", r["respuesta_final"])

if SAVE:
    E.to_csv(f"{RUTA}/specs_train_v0.csv", index=False)
    shutil.copy(DB_LOCAL, DB_DRIVE)
    print("\nGuardado: specs_train_v0.csv y caché en Drive")
else:
    print("\nSAVE = False: no se guardó nada")

In [ ]:
print("CELDA: DIAGNÓSTICO DE ESPECIFICACIONES (v0)")
assert all(x in globals() for x in ["E", "c", "datos"]), "Ejecutá antes la celda de especificaciones"
import json
import pandas as pd

# 1) Concentración por familia
fam = E.groupby("familia").agg(especificaciones=("tipo", "size"), series=("serie_id", "nunique"))
fam["pct"] = (fam["especificaciones"] / len(E) * 100).round(1)
print("Familias con más especificaciones:")
print(fam.sort_values("especificaciones", ascending=False).head(8).to_string())
print("Las 3 primeras suman:", round(fam["pct"].sort_values(ascending=False).head(3).sum(), 1), "%")

# 2) Variación interanual: valores crudos de la ventana (t y un año antes)
V = E[E["tipo"] == "variacion_interanual"]
filas = []
for _, r in V.iterrows():
    p = json.loads(r["params"])
    raw = c.get({"ids": p["ids"], "start_date": p["start_date"], "end_date": p["end_date"], "limit": 1000})
    d = dict(datos(raw) or [])
    filas.append({"serie": r["serie_titulo"], "freq": r["frecuencia"],
                  "v_prev": d.get(p["start_date"]), "v_t": d.get(p["end_date"]),
                  "variacion": json.loads(r["data"])[-1][1]})
W = pd.DataFrame(filas)

no_pos = (W["v_prev"].fillna(0) <= 0) | (W["v_t"].fillna(0) <= 0)
print("\nVariaciones con algún valor crudo <= 0 o faltante:", int(no_pos.sum()), "de", len(W))
print("Variaciones menores a -100%:", int((W["variacion"] < -1).sum()))
print("\nCasos sospechosos (valor crudo no positivo, menor a -100% o mayor a 200%):")
raros = W[no_pos | (W["variacion"] < -1) | (W["variacion"] > 2)]
print(raros.round(3).head(12).to_string())
print("\nCaché:", c.stats, "| entradas:", c.tamano())

In [ ]:
import os, json, time, shutil, random, zlib, sqlite3, threading
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import pandas as pd
import requests
from google.colab import drive

print("CELDA: ESPECIFICACIONES v1 (FILTRO DE POSITIVIDAD, CUOTA POR FAMILIA, 4 HILOS)")
drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
BASE = "https://apis.datos.gob.ar/series/api/series"
HEADERS = {"User-Agent": "Mozilla/5.0"}
DB_DRIVE = f"{RUTA}/cache_api.db"
DB_LOCAL = "/content/cache_api.db"

SAVE = True          # seguro: crea un archivo nuevo y la caché es de solo agregado
N_SERIES = 300       # series con datos
N_SIN_DATOS = 40     # series sin datos
CAP_SERIES_FAM = 10  # máximo de series por familia
MAX_VAR = 5.0        # variaciones mayores a 500% se descartan
HILOS = 4
LAG = {"mensual": 12, "trimestral": 4, "anual": 1}   # diarias y semestrales: fuera de esta versión

if os.path.exists(DB_DRIVE) and not os.path.exists(DB_LOCAL):
    shutil.copy(DB_DRIVE, DB_LOCAL)
    print("Caché copiada desde Drive")

# ---------- Caché segura para varios hilos ----------
PARAMS_OK = {"ids", "start_date", "end_date", "limit", "start", "sort",
             "collapse", "collapse_aggregation", "representation_mode"}

def clave(params):
    extra = set(params) - PARAMS_OK
    if extra:
        raise ValueError(f"Parámetros no previstos: {sorted(extra)}")
    p = {k: str(v) for k, v in params.items() if v is not None}
    return json.dumps(dict(sorted(p.items())), ensure_ascii=False)

class CachePar:
    def __init__(self, ruta_db, pausa=0.3):
        self.con = sqlite3.connect(ruta_db, check_same_thread=False)
        self.con.execute("CREATE TABLE IF NOT EXISTS cache (clave TEXT PRIMARY KEY, "
                         "status INTEGER, cuerpo TEXT, consultado TEXT)")
        self.con.commit()
        self.lock = threading.Lock()
        self.pausa = pausa
        self.stats = {"aciertos": 0, "fallos_de_cache": 0, "error_de_red": 0}

    def get(self, params):
        k = clave(params)
        with self.lock:
            fila = self.con.execute("SELECT status, cuerpo, consultado FROM cache WHERE clave=?",
                                    (k,)).fetchone()
            self.stats["aciertos" if fila else "fallos_de_cache"] += 1
        if fila:
            return {"status": fila[0], "json": json.loads(fila[1]),
                    "consultado": fila[2], "de_cache": True}
        for intento in range(3):
            try:
                r = requests.get(BASE, params=json.loads(k), headers=HEADERS, timeout=30)
                time.sleep(self.pausa)
                if r.status_code in (200, 400):
                    js = r.json()
                    ahora = datetime.now(timezone.utc).isoformat(timespec="seconds")
                    with self.lock:
                        self.con.execute("INSERT OR REPLACE INTO cache VALUES (?,?,?,?)",
                                         (k, r.status_code, json.dumps(js, ensure_ascii=False), ahora))
                        self.con.commit()
                    return {"status": r.status_code, "json": js, "consultado": ahora, "de_cache": False}
            except Exception:
                pass
            time.sleep(1 + intento)
        with self.lock:
            self.stats["error_de_red"] += 1
        return {"status": None, "json": None, "consultado": None, "de_cache": False}

    def tamano(self):
        with self.lock:
            return self.con.execute("SELECT COUNT(*) FROM cache").fetchone()[0]

c2 = CachePar(DB_LOCAL)
print("Entradas en la caché al empezar:", c2.tamano())

# ---------- Auxiliares ----------
def datos(resp):
    return (resp["json"] or {}).get("data")

def periodo(fecha, freq):
    return fecha[:4] if freq == "anual" else fecha[:7]

def desplazar(fecha, anios):
    return (pd.to_datetime(fecha) + pd.DateOffset(years=anios)).strftime("%Y-%m-%d")

def periodo_al_azar(rng, freq):
    anio = rng.randint(2015, 2024)
    if freq == "anual":
        return str(anio)
    mes = rng.choice([1, 4, 7, 10]) if freq == "trimestral" else rng.randint(1, 12)
    return f"{anio}-{mes:02d}"

def fmt(v):
    return f"{v:.4f}".rstrip("0").rstrip(".")

def nombre(row):
    return str(row["serie_titulo"]).replace("_", " ")

def unidad(row):
    u = row["serie_unidades"]
    return f" ({u})" if isinstance(u, str) and u.strip() else ""

def fila_spec(tipo, row, params, resp, final, caso=""):
    return {"tipo": tipo, "caso": caso, "serie_id": row["serie_id"], "familia": row["familia"],
            "frecuencia": row["frecuencia"], "serie_titulo": row["serie_titulo"],
            "serie_unidades": row["serie_unidades"],
            "params": json.dumps(params, ensure_ascii=False, sort_keys=True),
            "status": resp["status"], "data": json.dumps(datos(resp), ensure_ascii=False),
            "consultado": resp["consultado"], "respuesta_final": final}

# ---------- Selección (solo TRAIN, con cuota por familia) ----------
df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
base = df[(df["split"] == "train") & df["frecuencia"].isin(LAG)]
con_datos = base[base["estado"].isin(["normal", "api_adelantada"])]
sin_datos = base[base["estado"] == "sin_datos"]

def elegir(d, n):
    return d.sample(frac=1, random_state=7).groupby("familia").head(CAP_SERIES_FAM).head(n)

S1, S0 = elegir(con_datos, N_SERIES), elegir(sin_datos, N_SIN_DATOS)
print(f"Series con datos: {len(S1)} (familias: {S1['familia'].nunique()}) | "
      f"sin datos: {len(S0)} (familias: {S0['familia'].nunique()})")

# ---------- Una serie con datos -> sus especificaciones ----------
def procesar(row):
    out, desc = [], Counter()
    sid, freq = row["serie_id"], row["frecuencia"]
    rng = random.Random(zlib.crc32(sid.encode()))
    nom, uni = nombre(row), unidad(row)

    r0 = c2.get({"ids": sid, "limit": 1, "sort": "asc"})
    rl = c2.get({"ids": sid, "limit": 1, "sort": "desc"})
    if not (r0["status"] == 200 and datos(r0)):
        desc["la serie no devolvió datos al verificar"] += 1
        return out, desc
    n = r0["json"]["count"]
    primera = datos(r0)[0][0]
    ultimo = datos(rl)[0] if (rl["status"] == 200 and datos(rl)) else None

    # ultimo_dato
    if ultimo and ultimo[1] is not None:
        out.append(fila_spec("ultimo_dato", row, {"ids": sid, "limit": 1, "sort": "desc"}, rl,
                             f"El último dato disponible de {nom}{uni} es {fmt(ultimo[1])}, "
                             f"de la fecha {ultimo[0]}."))
    else:
        desc["ultimo_dato: respuesta vacía o nula"] += 1

    # valor_puntual
    pt = datos(c2.get({"ids": sid, "limit": 1, "sort": "asc", "start": rng.randrange(n)}))
    if pt:
        p = periodo(pt[0][0], freq)
        params = {"ids": sid, "start_date": p, "end_date": p}
        rr = c2.get(params)
        d = datos(rr)
        if rr["status"] == 200 and d and d[0][1] is not None:
            out.append(fila_spec("valor_puntual", row, params, rr, f"{nom}{uni}: {fmt(d[0][1])} en {p}."))
        else:
            desc["valor_puntual: sin valor"] += 1

    # variacion_interanual: ventana que incluye el año previo + filtro de positividad
    lag = LAG[freq]
    if n > lag:
        pt = datos(c2.get({"ids": sid, "limit": 1, "sort": "asc", "start": rng.randrange(lag, n)}))
        if pt:
            d0 = pt[0][0]
            prev = desplazar(d0, -1)
            crudo = dict(datos(c2.get({"ids": sid, "start_date": prev, "end_date": d0, "limit": 1000})) or [])
            v_t, v_prev = crudo.get(d0), crudo.get(prev)
            params = {"ids": sid, "start_date": prev, "end_date": d0,
                      "representation_mode": "percent_change_a_year_ago"}
            rr = c2.get(params)
            d = datos(rr)
            if not (rr["status"] == 200 and d and d[-1][0] == d0 and d[-1][1] is not None):
                desc["variacion: sin dato interanual"] += 1
            elif v_t is None or v_prev is None or v_t <= 0 or v_prev <= 0:
                desc["variacion: valor crudo no positivo o faltante"] += 1
            else:
                v = d[-1][1]
                if abs(v - (v_t / v_prev - 1)) > 1e-6:
                    desc["variacion: la API no coincide con el cálculo propio"] += 1
                elif abs(v) > MAX_VAR:
                    desc["variacion: valor absurdo (> 500%)"] += 1
                else:
                    out.append(fila_spec("variacion_interanual", row, params, rr,
                                         f"La variación interanual de {nom} en {periodo(d0, freq)} fue de "
                                         f"{v * 100:.2f}% (respecto de {periodo(prev, freq)})."))

    # sin_datos: período fuera del rango de la serie
    antes = (ultimo is None) or rng.random() < 0.5
    fuera = desplazar(primera, -3) if antes else desplazar(ultimo[0], 2)
    p = periodo(fuera, freq)
    params = {"ids": sid, "start_date": p, "end_date": p}
    rr = c2.get(params)
    if rr["status"] == 200 and datos(rr) == []:
        out.append(fila_spec("sin_datos", row, params, rr,
                             f"La API no devuelve datos de {nom} para el período {p}.",
                             caso="fuera_de_rango_antes" if antes else "fuera_de_rango_despues"))
    else:
        desc["sin_datos: la API devolvió datos o error"] += 1
    return out, desc

# ---------- Una serie que la API devuelve vacía ----------
def procesar_vacia(row):
    out, desc = [], Counter()
    rng = random.Random(zlib.crc32(row["serie_id"].encode()))
    p = periodo_al_azar(rng, row["frecuencia"])
    params = {"ids": row["serie_id"], "start_date": p, "end_date": p}
    rr = c2.get(params)
    if rr["status"] == 200 and datos(rr) == []:
        out.append(fila_spec("sin_datos", row, params, rr,
                             f"La API no devuelve datos para la serie {nombre(row)}.", caso="serie_sin_datos"))
    else:
        desc["serie_sin_datos: la API devolvió datos o error"] += 1
    return out, desc

# ---------- Ejecución en paralelo ----------
t0 = time.time()
specs, descartes = [], Counter()
with ThreadPoolExecutor(max_workers=HILOS) as ex:
    for k, (o, d) in enumerate(ex.map(procesar, S1.to_dict("records")), 1):
        specs += o
        descartes += d
        if k % 25 == 0:
            print(f"[{k}/{len(S1)}] {time.time() - t0:.0f}s | specs: {len(specs)}", flush=True)
    for o, d in ex.map(procesar_vacia, S0.to_dict("records")):
        specs += o
        descartes += d

assert len(specs) > 0, "No se generó ninguna especificación"
E1 = pd.DataFrame(specs)

# ---------- Resumen ----------
print(f"\nEspecificaciones: {len(E1)} | series distintas: {E1['serie_id'].nunique()} | "
      f"familias: {E1['familia'].nunique()}")
print(E1.groupby(["tipo", "caso"]).size().to_string())
print("Descartes:", dict(descartes))
print("Caché:", c2.stats, "| entradas:", c2.tamano(), f"| {time.time() - t0:.0f}s")

fam = E1["familia"].value_counts(normalize=True).mul(100).round(1)
print("\nFamilias con más especificaciones (%):", fam.head(5).to_dict())
print("Las 3 primeras suman:", round(fam.head(3).sum(), 1), "%")

v3 = E1[E1["tipo"] == "variacion_interanual"]["data"].map(lambda s: json.loads(s)[-1][1])
if len(v3):
    print("\nVariación interanual (fracción):")
    print(v3.describe().round(3).to_string())

for tipo in E1["tipo"].unique():
    r = E1[E1["tipo"] == tipo].iloc[0]
    print(f"\n--- {tipo}\n  serie: {r['serie_titulo']} | {r['frecuencia']}\n  consulta: {r['params']}"
          f"\n  respuesta final: {r['respuesta_final']}")

# ---------- Guardar ----------
if SAVE:
    c2.con.commit()
    E1.to_csv(f"{RUTA}/specs_train_v1.csv", index=False)
    shutil.copy(DB_LOCAL, DB_DRIVE)
    print("\nGuardado: specs_train_v1.csv y caché en Drive")
else:
    print("\nSAVE = False: no se guardó nada")

In [ ]:
!pip -q install -U anthropic
import os, re, json, time, unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import anthropic
from google.colab import drive, userdata

print("CELDA: PREGUNTAS (HAIKU) v1b - COMPLETO")
drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
PARCIAL = f"{RUTA}/preguntas_train_v1_parcial.csv"
SALIDA = f"{RUTA}/preguntas_train_v1.csv"
N_MAX = None        # None = todas las especificaciones
HILOS = 4
MODEL = "claude-haiku-5-5"
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=3, timeout=90.0)

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

def g(c, col):
    v = c[col]
    return "" if pd.isna(v) else str(v)

MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "octubre", "noviembre", "diciembre"]
TRIM = ["primer", "segundo", "tercer", "cuarto"]
PAT_INTERANUAL = r"interanual|\bano (anterior|pasado|previo)|mismo (mes|trimestre|periodo)|un ano (antes|atras)|hace un ano"

E = pd.read_csv(f"{RUTA}/specs_train_v1.csv")
E["spec_id"] = E.index
cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id"))
orden = E.sample(frac=1, random_state=1)
if N_MAX:
    orden = orden.head(N_MAX)

# ---------- Período de cada especificación ----------
def periodo_de_spec(r):
    if r["tipo"] == "ultimo_dato":
        return None
    p = json.loads(r["params"])
    if r["tipo"] == "variacion_interanual":
        e = p["end_date"]
        return e[:4] if r["frecuencia"] == "anual" else e[:7]
    return p["start_date"]

def periodo_humano(freq, p):
    if freq == "anual":
        return p[:4]
    y, m = p[:4], int(p[5:7])
    if freq == "mensual":
        return f"{MESES[m - 1]} de {y}"
    return f"{TRIM[(m - 1) // 3]} trimestre de {y}"

# ---------- Validación por código ----------
def validar(r, q, estilo, p):
    if not isinstance(q, str) or not q.strip():
        return "vacía"
    n = len(q.split())
    lo, hi = (3, 14) if estilo == "coloquial" else (8, 45)
    if not lo <= n <= hi:
        return f"largo ({n} palabras)"
    t = norm(q)
    anios = {int(a) for a in re.findall(r"\b(?:19|20)\d{2}\b", t)}
    if p is None:
        if re.search(r"\d", t):
            return "tiene números y no debería"
    else:
        y = int(p[:4])
        permitidos = {y, y - 1} if r["tipo"] == "variacion_interanual" else {y}
        if y not in anios:
            return "falta el año"
        if not anios <= permitidos:
            return "año de más"
        if re.search(r"\d", re.sub(r"\b(?:19|20)\d{2}\b", "", t)):
            return "tiene otros números"
        if r["frecuencia"] == "mensual":
            m = int(p[5:7])
            nombres = [MESES[m - 1]] + (["setiembre"] if m == 9 else [])
            if not any(x in t for x in nombres):
                return "falta el mes"
        if r["frecuencia"] == "trimestral" and "trimestre" not in t:
            return "falta 'trimestre'"
    titulo = norm(r["serie_titulo"])
    umbral = 3 if estilo == "coloquial" else 5
    if len(titulo.split()) >= umbral and titulo in t:
        return "copia el título"
    if r["tipo"] == "variacion_interanual" and not re.search(PAT_INTERANUAL, t):
        return "no aclara que es interanual"
    return ""

# ---------- Prompt y llamada ----------
def armar_prompt(r, p):
    c = cat.loc[r["serie_id"]]
    if r["tipo"] == "ultimo_dato":
        pedido = "el dato más reciente disponible."
        regla = "No menciones ninguna fecha, año ni período."
    else:
        hp = periodo_humano(r["frecuencia"], p)
        if r["tipo"] == "variacion_interanual":
            pedido = f"cuánto varió respecto del mismo período del año anterior (variación interanual) en {hp}."
            regla = (f"Mencioná este período: {hp}. Escribí el año en cifras y el mes o el trimestre en palabras. "
                     "Podés mencionar también el año anterior. En ambas versiones dejá explícito que es una "
                     "variación interanual o respecto del año anterior.")
        else:
            pedido = f"el valor del dato en {hp}."
            regla = (f"Mencioná este período: {hp}. Escribí el año en cifras y el mes o el trimestre en palabras. "
                     "No menciones ningún otro año.")
    return f"""Sos un usuario argentino que quiere consultar un dato estadístico y no conoce los nombres técnicos de la base de datos.
Escribí DOS versiones de la misma pregunta:
- "coloquial": entre 4 y 12 palabras, como escribiría alguien rápido en un chat. Solo el tema principal, sin detalles técnicos.
- "tecnica": entre 10 y 35 palabras, más precisa, con el detalle necesario para distinguir la serie de otras parecidas (frecuencia, unidad, base u otra característica).

La pregunta pide: {pedido}

Reglas:
- {regla}
- No agregues otros números.
- No copies el título técnico ni uses guiones bajos.

Devolvé solo JSON: {{"coloquial": "...", "tecnica": "..."}}

Datos de la serie:
Título técnico: {g(c, 'serie_titulo')}
Descripción: {g(c, 'serie_descripcion')}
Unidades: {g(c, 'serie_unidades')}
Frecuencia: {g(c, 'frecuencia')}
Dataset: {g(c, 'dataset_titulo')}
Distribución: {g(c, 'distribucion_titulo')}"""

def llamar(prompt, max_tokens):
    msg = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                 messages=[{"role": "user", "content": prompt}])
    txt = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
    return txt, msg.stop_reason == "max_tokens", msg.usage.input_tokens, msg.usage.output_tokens

def generar(r):
    p = periodo_de_spec(r)
    prompt = armar_prompt(r, p)
    motivos, ent, sal = [], 0, 0
    for intento in range(3):
        mt = 3000 if intento < 2 else 6000
        try:
            txt, truncado, e, s = llamar(prompt, mt)
        except Exception as ex:
            return {"ok": False, "coloquial": "", "tecnica": "", "motivos": " | ".join(motivos),
                    "error": f"{type(ex).__name__}: {str(ex)[:100]}", "intentos": intento + 1,
                    "ent": ent, "sal": sal}
        ent += e
        sal += s
        if truncado:
            motivos.append("max_tokens")
            continue
        try:
            js = json.loads(txt[txt.find("{"): txt.rfind("}") + 1])
            col, tec = js["coloquial"], js["tecnica"]
        except Exception:
            motivos.append("JSON inválido")
            continue
        m1, m2 = validar(r, col, "coloquial", p), validar(r, tec, "tecnica", p)
        if not m1 and not m2:
            return {"ok": True, "coloquial": col, "tecnica": tec, "motivos": " | ".join(motivos),
                    "error": "", "intentos": intento + 1, "ent": ent, "sal": sal}
        if m1:
            motivos.append(f"coloquial: {m1}")
        if m2:
            motivos.append(f"tecnica: {m2}")
    return {"ok": False, "coloquial": "", "tecnica": "", "motivos": " | ".join(motivos),
            "error": "sin respuesta válida tras 3 intentos", "intentos": 3, "ent": ent, "sal": sal}

def tarea(r):
    out = generar(r)
    out["spec_id"] = r["spec_id"]
    return out

# ---------- Retomar: se conservan solo las filas que salieron bien ----------
prev = pd.read_csv(PARCIAL) if os.path.exists(PARCIAL) else pd.DataFrame()
if len(prev):
    prev = prev[prev["ok"] == True]
hechos = set(prev["spec_id"]) if len(prev) else set()
pend = [r for r in orden.to_dict("records") if r["spec_id"] not in hechos]
print(f"Especificaciones: {len(orden)} | ya hechas: {len(hechos)} | pendientes: {len(pend)}")

t0, res = time.time(), []
with ThreadPoolExecutor(max_workers=HILOS) as ex:
    for k, out in enumerate(ex.map(tarea, pend), 1):
        res.append(out)
        if k % 50 == 0:
            pd.concat([prev, pd.DataFrame(res)], ignore_index=True).to_csv(PARCIAL, index=False)
            print(f"[{k}/{len(pend)}] {time.time() - t0:.0f}s", flush=True)

R = pd.DataFrame(res)
todo = pd.concat([prev, R], ignore_index=True) if len(R) else prev
todo.to_csv(PARCIAL, index=False)

# ---------- Resumen ----------
if len(R):
    costo = R["ent"].sum() * 0.10 / 1e6 + R["sal"].sum() * 0.50 / 1e6
    print(f"\nEsta corrida: {int(R['ok'].sum())}/{len(R)} ok | costo ${costo:.4f} | {time.time() - t0:.0f}s")
    print(f"Necesitaron más de un intento: {(R['intentos'] > 1).mean():.0%}")
    mot = Counter()
    for m in R["motivos"].fillna(""):
        for x in [y for y in m.split(" | ") if y]:
            mot[re.sub(r"\s*\(.*\)", "", x)] += 1
    print("Rechazos (incluye los recuperados en un reintento):", dict(mot.most_common(8)))
    print("Fallos definitivos:", Counter(R[~R["ok"]]["error"]).most_common(3))
    R2 = R.merge(E[["spec_id", "tipo"]], on="spec_id")
    print("\nTasa de éxito por tipo (esta corrida):")
    print(R2.groupby("tipo")["ok"].agg(n="size", tasa_ok="mean").round(2).to_string())

ok = todo[todo["ok"] == True].merge(
    E[["spec_id", "tipo", "caso", "serie_id", "familia", "frecuencia", "params"]], on="spec_id")
print("\nPreguntas válidas:", len(ok), "de", len(orden), "| por tipo:", ok["tipo"].value_counts().to_dict())
print("Series distintas:", ok["serie_id"].nunique(), "| familias:", ok["familia"].nunique())
print("Palabras (prom.): coloquial", round(ok["coloquial"].str.split().str.len().mean(), 1),
      "| técnica", round(ok["tecnica"].str.split().str.len().mean(), 1))

for tipo in ["variacion_interanual", "sin_datos"]:
    print(f"\n--- {tipo}")
    for _, r in ok[ok["tipo"] == tipo].sample(3, random_state=2).iterrows():
        print(f"  COLOQUIAL: {r['coloquial']}")
        print(f"  TÉCNICA:   {r['tecnica']}")

ok.drop(columns=["ok", "error", "ent", "sal"]).to_csv(SALIDA, index=False)
print("\nGuardado:", SALIDA)

In [ ]:
print("CELDA: DIAGNÓSTICO DE PREGUNTAS (SIN COSTO)")
assert all(x in globals() for x in ["E", "ok", "validar", "periodo_de_spec", "norm", "PARCIAL"]), \
    "Ejecutá antes la celda de preguntas v1b (define E, ok, validar, periodo_de_spec, norm y PARCIAL)"
import pandas as pd

APLICAR = False   # True: saca del parcial las preguntas que no pasan, para regenerarlas

Em = E.set_index("spec_id")

# 1) Re-validar TODAS las preguntas con las reglas actuales
fallan = []
for _, r in ok.iterrows():
    s = Em.loc[r["spec_id"]]
    p = periodo_de_spec(s)
    for estilo in ["coloquial", "tecnica"]:
        m = validar(s, r[estilo], estilo, p)
        if m:
            fallan.append({"spec_id": r["spec_id"], "tipo": s["tipo"], "estilo": estilo,
                           "motivo": m, "pregunta": r[estilo]})
F = pd.DataFrame(fallan)
print("Preguntas que no pasan las reglas actuales:", len(F),
      "| especificaciones afectadas:", F["spec_id"].nunique() if len(F) else 0)
if len(F):
    print(F["motivo"].value_counts().to_string())
    print(F.head(8)[["tipo", "estilo", "motivo", "pregunta"]].to_string(index=False))

# 2) Períodos en el futuro (hoy es noviembre de 2026)
def anio(r):
    p = periodo_de_spec(Em.loc[r["spec_id"]])
    return int(p[:4]) if p else None

ok2 = ok.copy()
ok2["anio_pedido"] = ok2.apply(anio, axis=1)
sd = ok2[ok2["tipo"] == "sin_datos"]
print("\nsin_datos:", len(sd), "| piden un año posterior a 2026:", int((sd["anio_pedido"] > 2026).sum()))
print("  por caso:", sd.groupby("caso")["anio_pedido"].agg(lambda s: int((s > 2026).sum())).to_dict())
print("Años pedidos por tipo (mín, mediana, máx):")
print(ok2.dropna(subset=["anio_pedido"]).groupby("tipo")["anio_pedido"].agg(["min", "median", "max"]).to_string())

# 3) Preguntas coloquiales repetidas
dup = ok["coloquial"].map(norm).value_counts()
print("\nPreguntas coloquiales que aparecen más de una vez:", int((dup > 1).sum()),
      "| filas involucradas:", int(dup[dup > 1].sum()))
print(dup[dup > 1].head(5).to_string())

# 4) Concentración por familia
fam = ok["familia"].value_counts(normalize=True).mul(100).round(1)
print("\nFamilias con más preguntas (%):", fam.head(5).to_dict(), "| las 3 primeras suman:", round(fam.head(3).sum(), 1), "%")

# 5) Regenerar las que no pasan (con APLICAR = True)
if APLICAR and len(F):
    par = pd.read_csv(PARCIAL)
    antes = len(par)
    par = par[~par["spec_id"].isin(set(F["spec_id"]))]
    par.to_csv(PARCIAL, index=False)
    print(f"\nSacadas del parcial: {antes - len(par)} filas. Ejecutá de nuevo la celda v1b para regenerarlas.")
else:
    print("\nAPLICAR = False: no se modificó nada")

In [ ]:
print("CELDA: CORRECCIÓN DE sin_datos (PERÍODOS PASADOS) + REGENERACIÓN PENDIENTE")
need = ["c2", "datos", "periodo", "desplazar", "RUTA", "PARCIAL", "norm", "PAT_INTERANUAL"]
assert all(x in globals() for x in need), \
    "Ejecutá antes las celdas de especificaciones v1 y de preguntas v1b (definen: " + ", ".join(need) + ")"
import json, random, re, shutil, zlib
from collections import Counter
import pandas as pd

SAVE = True                                 # primero False; revisar; después True (una sola vez)
PERIODOS_HASTA = pd.Timestamp("2026-06-01")    # último período permitido (inicio del período)
ARCH = f"{RUTA}/specs_train_v1.csv"
PASO = {"mensual": 1, "trimestral": 3, "anual": 12}

def sumar(fecha, freq, k):
    return pd.to_datetime(fecha) + pd.DateOffset(months=PASO[freq] * k)

E1 = pd.read_csv(ARCH)                          # el índice de la tabla es el spec_id
log, cambio_periodo, nuevas = Counter(), [], {}

for i, r in E1[E1["caso"].isin(["fuera_de_rango_antes", "fuera_de_rango_despues"])].iterrows():
    sid, freq = r["serie_id"], r["frecuencia"]
    rng = random.Random(zlib.crc32(sid.encode()) + 1)
    primera = datos(c2.get({"ids": sid, "limit": 1, "sort": "asc"}))[0][0]
    ultimo = datos(c2.get({"ids": sid, "limit": 1, "sort": "desc"}))[0][0]
    p_viejo = json.loads(r["params"])["start_date"]
    caso, p = r["caso"], p_viejo

    if caso == "fuera_de_rango_despues":
        cand = [d for d in (sumar(ultimo, freq, k) for k in range(1, 7)) if d <= PERIODOS_HASTA]
        if cand:
            p = periodo(rng.choice(cand).strftime("%Y-%m-%d"), freq)
        else:
            caso = "fuera_de_rango_antes"
            p = periodo(desplazar(primera, -3), freq)
            log["pasó de 'después' a 'antes' (la serie termina muy cerca de hoy)"] += 1

    params = {"ids": sid, "start_date": p, "end_date": p}
    resp = c2.get(params)
    if not (resp["status"] == 200 and datos(resp) == []):
        log["se conserva la original (el período nuevo no devolvió vacío)"] += 1
        continue
    nom = str(r["serie_titulo"]).replace("_", " ")
    nuevas[i] = {"params": json.dumps(params, ensure_ascii=False, sort_keys=True),
                 "data": json.dumps(datos(resp), ensure_ascii=False), "status": resp["status"],
                 "consultado": resp["consultado"], "caso": caso,
                 "respuesta_final": f"La API no devuelve datos de {nom} para el período {p}. "
                                    f"La serie tiene datos entre {periodo(primera, freq)} y {periodo(ultimo, freq)}."}
    if p != p_viejo:
        cambio_periodo.append(i)

for i, vals in nuevas.items():
    for col, v in vals.items():
        E1.at[i, col] = v
print("Especificaciones fuera_de_rango actualizadas:", len(nuevas), "| con período nuevo:", len(cambio_periodo))
print(dict(log))

sd = E1[E1["tipo"] == "sin_datos"].copy()
sd["anio"] = sd["params"].map(lambda s: int(json.loads(s)["start_date"][:4]))
print("\nsin_datos por caso:", sd["caso"].value_counts().to_dict())
print("sin_datos que piden un año posterior a 2026:", int((sd["anio"] > 2026).sum()), "de", len(sd))
print("\nEjemplos:")
for _, r in sd[sd["caso"].str.startswith("fuera")].sample(3, random_state=2).iterrows():
    print(" ", r["caso"], "|", r["params"][:85], "\n   ", r["respuesta_final"])

# ---------- Qué preguntas hay que regenerar ----------
par = pd.read_csv(PARCIAL)
mal = set()
for _, q in par[par["ok"] == True].iterrows():
    if E1.at[q["spec_id"], "tipo"] == "variacion_interanual" and not re.search(PAT_INTERANUAL, norm(q["coloquial"])):
        mal.add(q["spec_id"])
regen = set(cambio_periodo) | mal
print(f"\nPreguntas a regenerar: {len(regen)} (período cambiado: {len(cambio_periodo)}, variación sin 'interanual': {len(mal)})")

if SAVE:
    shutil.copy(ARCH, f"{RUTA}/specs_train_v1_antes_de_corregir.csv")
    E1.to_csv(ARCH, index=False)
    par[~par["spec_id"].isin(regen)].to_csv(PARCIAL, index=False)
    print("Guardado. Respaldo: specs_train_v1_antes_de_corregir.csv. Ahora ejecutá de nuevo la celda v1b.")
else:
    print("SAVE = False: no se modificó nada")

In [ ]:
print("CELDA: ESPECIFICACIONES v2 PARA VALIDACIÓN Y TEST")
need = ["c2", "datos", "periodo", "desplazar", "fmt", "nombre", "unidad", "fila_spec",
        "procesar_vacia", "LAG", "MAX_VAR", "RUTA", "DB_LOCAL", "DB_DRIVE"]
assert all(x in globals() for x in need), \
    "Ejecutá antes la celda de especificaciones v1 (define: " + ", ".join(need) + ")"
import json, random, shutil, time, zlib
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

SAVE = True          # seguro: crea archivos nuevos; la caché es de solo agregado
HILOS = 4
CAP_SERIES_FAM = 10
PERIODOS_HASTA = pd.Timestamp("2026-06-01")
CUPOS = {"val": (80, 10), "test": (150, 20)}      # (series con datos, series sin datos)
PASO = {"mensual": 1, "trimestral": 3, "anual": 12}

def sumar(fecha, freq, k):
    return pd.to_datetime(fecha) + pd.DateOffset(months=PASO[freq] * k)

def procesar_eval(row):
    out, desc = [], Counter()
    sid, freq = row["serie_id"], row["frecuencia"]
    rng = random.Random(zlib.crc32(sid.encode()))
    nom, uni = nombre(row), unidad(row)

    r0 = c2.get({"ids": sid, "limit": 1, "sort": "asc"})
    rl = c2.get({"ids": sid, "limit": 1, "sort": "desc"})
    if not (r0["status"] == 200 and datos(r0)):
        desc["la serie no devolvió datos al verificar"] += 1
        return out, desc
    n = r0["json"]["count"]
    primera = datos(r0)[0][0]
    ultimo = datos(rl)[0] if (rl["status"] == 200 and datos(rl)) else None
    if ultimo is None:                                   # respaldo si sort=desc devuelve vacío
        alt = datos(c2.get({"ids": sid, "limit": 1, "sort": "asc", "start": n - 1}))
        ultimo = alt[0] if alt else None
    if ultimo is None or ultimo[1] is None:
        desc["no se pudo leer el último dato"] += 1
        return out, desc

    # ultimo_dato
    out.append(fila_spec("ultimo_dato", row, {"ids": sid, "limit": 1, "sort": "desc"}, rl,
                         f"El último dato disponible de {nom}{uni} es {fmt(ultimo[1])}, de la fecha {ultimo[0]}."))

    # valor_puntual
    pt = datos(c2.get({"ids": sid, "limit": 1, "sort": "asc", "start": rng.randrange(n)}))
    if pt:
        p = periodo(pt[0][0], freq)
        params = {"ids": sid, "start_date": p, "end_date": p}
        rr = c2.get(params)
        d = datos(rr)
        if rr["status"] == 200 and d and d[0][1] is not None:
            out.append(fila_spec("valor_puntual", row, params, rr, f"{nom}{uni}: {fmt(d[0][1])} en {p}."))
        else:
            desc["valor_puntual: sin valor"] += 1

    # variacion_interanual
    lag = LAG[freq]
    if n > lag:
        pt = datos(c2.get({"ids": sid, "limit": 1, "sort": "asc", "start": rng.randrange(lag, n)}))
        if pt:
            d0 = pt[0][0]
            prev = desplazar(d0, -1)
            crudo = dict(datos(c2.get({"ids": sid, "start_date": prev, "end_date": d0, "limit": 1000})) or [])
            v_t, v_prev = crudo.get(d0), crudo.get(prev)
            params = {"ids": sid, "start_date": prev, "end_date": d0,
                      "representation_mode": "percent_change_a_year_ago"}
            rr = c2.get(params)
            d = datos(rr)
            if not (rr["status"] == 200 and d and d[-1][0] == d0 and d[-1][1] is not None):
                desc["variacion: sin dato interanual"] += 1
            elif v_t is None or v_prev is None or v_t <= 0 or v_prev <= 0:
                desc["variacion: valor crudo no positivo o faltante"] += 1
            else:
                v = d[-1][1]
                if abs(v - (v_t / v_prev - 1)) > 1e-6:
                    desc["variacion: la API no coincide con el cálculo propio"] += 1
                elif abs(v) > MAX_VAR:
                    desc["variacion: valor absurdo (> 500%)"] += 1
                else:
                    out.append(fila_spec("variacion_interanual", row, params, rr,
                                         f"La variación interanual de {nom} en {periodo(d0, freq)} fue de "
                                         f"{v * 100:.2f}% (respecto de {periodo(prev, freq)})."))

    # sin_datos: período pasado, fuera del rango de la serie
    caso = "fuera_de_rango_despues" if rng.random() < 0.5 else "fuera_de_rango_antes"
    p = None
    if caso == "fuera_de_rango_despues":
        cand = [d for d in (sumar(ultimo[0], freq, k) for k in range(1, 7))
                if d <= PERIODOS_HASTA and (freq != "anual" or d.year <= 2025)]
        if cand:
            p = periodo(rng.choice(cand).strftime("%Y-%m-%d"), freq)
        else:
            caso = "fuera_de_rango_antes"
    if p is None:
        p = periodo(desplazar(primera, -rng.choice([1, 2, 3, 5])), freq)
    params = {"ids": sid, "start_date": p, "end_date": p}
    rr = c2.get(params)
    if rr["status"] == 200 and datos(rr) == []:
        out.append(fila_spec("sin_datos", row, params, rr,
                             f"La API no devuelve datos de {nom} para el período {p}. "
                             f"La serie tiene datos entre {periodo(primera, freq)} y {periodo(ultimo[0], freq)}.",
                             caso=caso))
    else:
        desc["sin_datos: la API devolvió datos o error"] += 1
    return out, desc

# ---------- Ejecución por split ----------
df = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
base = df[df["frecuencia"].isin(LAG)]

def elegir(d, n):
    return d.sample(frac=1, random_state=7).groupby("familia").head(CAP_SERIES_FAM).head(n)

resultados = {}
for split, (n1, n0) in CUPOS.items():
    b = base[base["split"] == split]
    S1 = elegir(b[b["estado"].isin(["normal", "api_adelantada"])], n1)
    S0 = elegir(b[b["estado"] == "sin_datos"], n0)
    print(f"\n=== {split}: series con datos {len(S1)} (familias {S1['familia'].nunique()}) | "
          f"sin datos {len(S0)} (familias {S0['familia'].nunique()})")
    t0, specs, descartes = time.time(), [], Counter()
    with ThreadPoolExecutor(max_workers=HILOS) as ex:
        for k, (o, d) in enumerate(ex.map(procesar_eval, S1.to_dict("records")), 1):
            specs += o
            descartes += d
            if k % 25 == 0:
                print(f"  [{k}/{len(S1)}] {time.time() - t0:.0f}s | specs: {len(specs)}", flush=True)
        for o, d in ex.map(procesar_vacia, S0.to_dict("records")):
            specs += o
            descartes += d
    assert specs, f"No se generaron especificaciones para {split}"
    E2 = pd.DataFrame(specs)
    E2["split"] = split
    resultados[split] = E2
    print(f"  Especificaciones: {len(E2)} | series: {E2['serie_id'].nunique()} | familias: {E2['familia'].nunique()}")
    print("  " + E2.groupby(["tipo", "caso"]).size().to_string().replace("\n", "\n  "))
    print("  Descartes:", dict(descartes))
    anios = E2[E2["tipo"] == "sin_datos"]["params"].map(lambda s: int(json.loads(s)["start_date"][:4]))
    print("  sin_datos con año posterior a 2026:", int((anios > 2026).sum()), "| tiempo:", f"{time.time() - t0:.0f}s")

print("\nCaché:", c2.stats, "| entradas:", c2.tamano())
if SAVE:
    c2.con.commit()
    for split, E2 in resultados.items():
        E2.to_csv(f"{RUTA}/specs_{split}_v2.csv", index=False)
    shutil.copy(DB_LOCAL, DB_DRIVE)
    print("Guardado: specs_val_v2.csv, specs_test_v2.csv y la caché en Drive")
else:
    print("SAVE = False: no se guardó nada")

In [ ]:
!pip -q install -U anthropic
import os, re, json, time, unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import anthropic
from google.colab import drive, userdata

print("CELDA: PREGUNTAS (HAIKU) PARA VALIDACIÓN Y TEST (v2)")
drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
SPLITS = ["train"]
HILOS = 4
MODEL = "claude-haiku-5-5"
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=3, timeout=90.0)

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

def g(c, col):
    v = c[col]
    return "" if pd.isna(v) else str(v)

MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "octubre", "noviembre", "diciembre"]
TRIM = ["primer", "segundo", "tercer", "cuarto"]
PAT_INTERANUAL = r"interanual|\bano (anterior|pasado|previo)|mismo (mes|trimestre|periodo)|un ano (antes|atras)|hace un ano"

cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id"))

def periodo_de_spec(r):
    if r["tipo"] == "ultimo_dato":
        return None
    p = json.loads(r["params"])
    if r["tipo"] == "variacion_interanual":
        e = p["end_date"]
        return e[:4] if r["frecuencia"] == "anual" else e[:7]
    return p["start_date"]

def periodo_humano(freq, p):
    if freq == "anual":
        return p[:4]
    y, m = p[:4], int(p[5:7])
    if freq == "mensual":
        return f"{MESES[m - 1]} de {y}"
    return f"{TRIM[(m - 1) // 3]} trimestre de {y}"

def validar(r, q, estilo, p):
    if not isinstance(q, str) or not q.strip():
        return "vacía"
    n = len(q.split())
    lo, hi = (3, 14) if estilo == "coloquial" else (8, 45)
    if not lo <= n <= hi:
        return f"largo ({n} palabras)"
    t = norm(q)
    anios = {int(a) for a in re.findall(r"\b(?:19|20)\d{2}\b", t)}
    if p is None:
        if re.search(r"\d", t):
            return "tiene números y no debería"
    else:
        y = int(p[:4])
        permitidos = {y, y - 1} if r["tipo"] == "variacion_interanual" else {y}
        if y not in anios:
            return "falta el año"
        if not anios <= permitidos:
            return "año de más"
        if re.search(r"\d", re.sub(r"\b(?:19|20)\d{2}\b", "", t)):
            return "tiene otros números"
        if r["frecuencia"] == "mensual":
            m = int(p[5:7])
            nombres = [MESES[m - 1]] + (["setiembre"] if m == 9 else [])
            if not any(x in t for x in nombres):
                return "falta el mes"
        if r["frecuencia"] == "trimestral" and "trimestre" not in t:
            return "falta 'trimestre'"
    titulo = norm(r["serie_titulo"])
    umbral = 3 if estilo == "coloquial" else 5
    if len(titulo.split()) >= umbral and titulo in t:
        return "copia el título"
    if r["tipo"] == "variacion_interanual" and not re.search(PAT_INTERANUAL, t):
        return "no aclara que es interanual"
    return ""

def armar_prompt(r, p):
    c = cat.loc[r["serie_id"]]
    if r["tipo"] == "ultimo_dato":
        pedido = "el dato más reciente disponible."
        regla = "No menciones ninguna fecha, año ni período."
    else:
        hp = periodo_humano(r["frecuencia"], p)
        if r["tipo"] == "variacion_interanual":
            pedido = f"cuánto varió respecto del mismo período del año anterior (variación interanual) en {hp}."
            regla = (f"Mencioná este período: {hp}. Escribí el año en cifras y el mes o el trimestre en palabras. "
                     "Podés mencionar también el año anterior. En ambas versiones dejá explícito que es una "
                     "variación interanual o respecto del año anterior.")
        else:
            pedido = f"el valor del dato en {hp}."
            regla = (f"Mencioná este período: {hp}. Escribí el año en cifras y el mes o el trimestre en palabras. "
                     "No menciones ningún otro año.")
    return f"""Sos un usuario argentino que quiere consultar un dato estadístico y no conoce los nombres técnicos de la base de datos.
Escribí DOS versiones de la misma pregunta:
- "coloquial": entre 4 y 12 palabras, como escribiría alguien rápido en un chat. Solo el tema principal, sin detalles técnicos.
- "tecnica": entre 10 y 35 palabras, más precisa, con el detalle necesario para distinguir la serie de otras parecidas (frecuencia, unidad, base u otra característica).

La pregunta pide: {pedido}

Reglas:
- {regla}
- No agregues otros números.
- No copies el título técnico ni uses guiones bajos.

Devolvé solo JSON: {{"coloquial": "...", "tecnica": "..."}}

Datos de la serie:
Título técnico: {g(c, 'serie_titulo')}
Descripción: {g(c, 'serie_descripcion')}
Unidades: {g(c, 'serie_unidades')}
Frecuencia: {g(c, 'frecuencia')}
Dataset: {g(c, 'dataset_titulo')}
Distribución: {g(c, 'distribucion_titulo')}"""

def llamar(prompt, max_tokens):
    msg = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                 messages=[{"role": "user", "content": prompt}])
    txt = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
    return txt, msg.stop_reason == "max_tokens", msg.usage.input_tokens, msg.usage.output_tokens

def generar(r):
    p = periodo_de_spec(r)
    prompt = armar_prompt(r, p)
    motivos, ent, sal = [], 0, 0
    for intento in range(3):
        mt = 3000 if intento < 2 else 6000
        try:
            txt, truncado, e, s = llamar(prompt, mt)
        except Exception as ex:
            return {"ok": False, "coloquial": "", "tecnica": "", "motivos": " | ".join(motivos),
                    "error": f"{type(ex).__name__}: {str(ex)[:100]}", "intentos": intento + 1,
                    "ent": ent, "sal": sal}
        ent += e
        sal += s
        if truncado:
            motivos.append("max_tokens")
            continue
        try:
            js = json.loads(txt[txt.find("{"): txt.rfind("}") + 1])
            col, tec = js["coloquial"], js["tecnica"]
        except Exception:
            motivos.append("JSON inválido")
            continue
        m1, m2 = validar(r, col, "coloquial", p), validar(r, tec, "tecnica", p)
        if not m1 and not m2:
            return {"ok": True, "coloquial": col, "tecnica": tec, "motivos": " | ".join(motivos),
                    "error": "", "intentos": intento + 1, "ent": ent, "sal": sal}
        if m1:
            motivos.append(f"coloquial: {m1}")
        if m2:
            motivos.append(f"tecnica: {m2}")
    return {"ok": False, "coloquial": "", "tecnica": "", "motivos": " | ".join(motivos),
            "error": "sin respuesta válida tras 3 intentos", "intentos": 3, "ent": ent, "sal": sal}

def tarea(r):
    out = generar(r)
    out["spec_id"] = r["spec_id"]
    return out

# ---------- Un split por vez, con retoma ----------
for split in SPLITS:
    parcial = f"{RUTA}/preguntas_{split}_v2_parcial.csv"
    salida = f"{RUTA}/preguntas_{split}_v2.csv"
    E = pd.read_csv(f"{RUTA}/specs_{split}_v2.csv")
    E["spec_id"] = [f"{split}-{i}" for i in E.index]

    prev = pd.read_csv(parcial) if os.path.exists(parcial) else pd.DataFrame()
    if len(prev):
        prev = prev[prev["ok"] == True]
    hechos = set(prev["spec_id"]) if len(prev) else set()
    pend = [r for r in E.to_dict("records") if r["spec_id"] not in hechos]
    print(f"\n=== {split}: {len(E)} especificaciones | ya hechas: {len(hechos)} | pendientes: {len(pend)}")

    t0, res = time.time(), []
    with ThreadPoolExecutor(max_workers=HILOS) as ex:
        for k, out in enumerate(ex.map(tarea, pend), 1):
            res.append(out)
            if k % 50 == 0:
                pd.concat([prev, pd.DataFrame(res)], ignore_index=True).to_csv(parcial, index=False)
                print(f"  [{k}/{len(pend)}] {time.time() - t0:.0f}s", flush=True)

    R = pd.DataFrame(res)
    todo = pd.concat([prev, R], ignore_index=True) if len(R) else prev
    todo.to_csv(parcial, index=False)

    if len(R):
        costo = R["ent"].sum() * 0.10 / 1e6 + R["sal"].sum() * 0.50 / 1e6
        print(f"  Esta corrida: {int(R['ok'].sum())}/{len(R)} ok | costo ${costo:.4f} | {time.time() - t0:.0f}s")
        mot = Counter()
        for m in R["motivos"].fillna(""):
            for x in [y for y in m.split(" | ") if y]:
                mot[re.sub(r"\s*\(.*\)", "", x)] += 1
        print("  Rechazos (incluye los recuperados en un reintento):", dict(mot.most_common(6)))
        print("  Fallos definitivos:", Counter(R[~R["ok"]]["error"]).most_common(3))

    ok = todo[todo["ok"] == True].merge(
        E[["spec_id", "tipo", "caso", "serie_id", "familia", "frecuencia", "params"]], on="spec_id")
    print(f"  Preguntas válidas: {len(ok)} de {len(E)} | por tipo: {ok['tipo'].value_counts().to_dict()}")
    print(f"  Palabras (prom.): coloquial {ok['coloquial'].str.split().str.len().mean():.1f} | "
          f"técnica {ok['tecnica'].str.split().str.len().mean():.1f}")
    dup = ok["coloquial"].map(norm).value_counts()
    print(f"  Coloquiales repetidas en más de una especificación: {int((dup > 1).sum())}")
    ok.drop(columns=["ok", "error", "ent", "sal"]).to_csv(salida, index=False)
    print("  Guardado:", salida)

In [ ]:
!pip -q install -U anthropic
import os, re, json, time, threading
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import anthropic
from google.colab import drive, userdata

print("CELDA: VERIFICADOR DE FIDELIDAD (VAL Y TEST)")
drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
SPLITS = ["val", "test"]
HILOS = 4
MODEL = "claude-haiku-5-5"
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=3, timeout=60.0)
ABORT = threading.Event()

cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id"))

def g(c, col):
    v = c[col]
    return "" if pd.isna(v) else str(v)

MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "octubre", "noviembre", "diciembre"]
TRIM = ["primer", "segundo", "tercer", "cuarto"]

def periodo_de_spec(r):
    if r["tipo"] == "ultimo_dato":
        return None
    p = json.loads(r["params"])
    if r["tipo"] == "variacion_interanual":
        e = p["end_date"]
        return e[:4] if r["frecuencia"] == "anual" else e[:7]
    return p["start_date"]

def periodo_humano(freq, p):
    if freq == "anual":
        return p[:4]
    y, m = p[:4], int(p[5:7])
    return f"{MESES[m - 1]} de {y}" if freq == "mensual" else f"{TRIM[(m - 1) // 3]} trimestre de {y}"

def operacion(r):
    p = periodo_de_spec(r)
    if r["tipo"] == "ultimo_dato":
        return "el dato más reciente disponible"
    hp = periodo_humano(r["frecuencia"], p)
    if r["tipo"] == "variacion_interanual":
        return f"la variación interanual en {hp}"
    return f"el valor en {hp}"

def armar_prompt(r):
    c = cat.loc[r["serie_id"]]
    return f"""Verificá si cada pregunta pide el dato descrito abajo, o si cambia su sentido (otra medida, otra entidad, otra operación u otro período).
Una pregunta coloquial puede ser más vaga que el dato, pero tiene que poder referirse razonablemente a ESE dato y no a uno claramente distinto.
Marcá "ok": false solo si hay un cambio de sentido real.

Dato real:
- Título técnico: {g(c, 'serie_titulo')}
- Descripción: {g(c, 'serie_descripcion')}
- Unidades: {g(c, 'serie_unidades')}
- Frecuencia: {g(c, 'frecuencia')}
- Dataset: {g(c, 'dataset_titulo')}
- Distribución: {g(c, 'distribucion_titulo')}
- Lo que se pide: {operacion(r)}

Pregunta coloquial: {r['coloquial']}
Pregunta técnica: {r['tecnica']}

Devolvé solo JSON: {{"coloquial": {{"ok": true, "motivo": ""}}, "tecnica": {{"ok": true, "motivo": ""}}}}"""

def tarea(r):
    if ABORT.is_set():
        return {"spec_id": r["spec_id"], "ok": False, "error": "abortado"}
    try:
        msg = client.messages.create(model=MODEL, max_tokens=500,
                                     messages=[{"role": "user", "content": armar_prompt(r)}],
                                     extra_body={"thinking": {"type": "disabled"}})
        txt = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
        js = json.loads(txt[txt.find("{"): txt.rfind("}") + 1])
        return {"spec_id": r["spec_id"], "ok": True, "error": "",
                "col_ok": bool(js["coloquial"]["ok"]), "col_motivo": str(js["coloquial"].get("motivo", "")),
                "tec_ok": bool(js["tecnica"]["ok"]), "tec_motivo": str(js["tecnica"].get("motivo", "")),
                "ent": msg.usage.input_tokens, "sal": msg.usage.output_tokens}
    except anthropic.AuthenticationError as e:
        ABORT.set()
        return {"spec_id": r["spec_id"], "ok": False, "error": f"AuthenticationError (se corta todo): {str(e)[:80]}"}
    except Exception as e:
        return {"spec_id": r["spec_id"], "ok": False, "error": f"{type(e).__name__}: {str(e)[:100]}"}

for split in SPLITS:
    parcial = f"{RUTA}/verificacion_fidelidad_{split}_parcial.csv"
    Q = pd.read_csv(f"{RUTA}/preguntas_{split}_v2.csv")
    Q["tipo_p"] = Q["tipo"]
    dup = Q["coloquial"].str.lower().value_counts()
    print(f"\n=== {split}: {len(Q)} especificaciones | coloquiales repetidas: {int((dup > 1).sum())}")
    for txt in dup[dup > 1].index[:6]:
        print("   -", txt, "->", Q[Q["coloquial"].str.lower() == txt]["serie_id"].tolist())

    prev = pd.read_csv(parcial) if os.path.exists(parcial) else pd.DataFrame()
    if len(prev):
        prev = prev[prev["ok"] == True]
    hechos = set(prev["spec_id"]) if len(prev) else set()
    pend = [r for r in Q.to_dict("records") if r["spec_id"] not in hechos]
    print(f"  ya hechas: {len(hechos)} | pendientes: {len(pend)}")

    t0, res = time.time(), []
    with ThreadPoolExecutor(max_workers=HILOS) as ex:
        for k, out in enumerate(ex.map(tarea, pend), 1):
            res.append(out)
            if k % 100 == 0:
                pd.concat([prev, pd.DataFrame(res)], ignore_index=True).to_csv(parcial, index=False)
                print(f"  [{k}/{len(pend)}] {time.time() - t0:.0f}s", flush=True)
    R = pd.DataFrame(res)
    todo = pd.concat([prev, R], ignore_index=True) if len(R) else prev
    todo.to_csv(parcial, index=False)

    if len(R):
        costo = R["ent"].fillna(0).sum() * 0.10 / 1e6 + R["sal"].fillna(0).sum() * 0.50 / 1e6
        print(f"  Esta corrida: {int(R['ok'].sum())}/{len(R)} ok | costo ${costo:.4f} | {time.time() - t0:.0f}s")
        print("  Fallos:", Counter(R[~R["ok"]]["error"]).most_common(3))
    if ABORT.is_set():
        print("  *** Se cortó por error de autenticación: revisá la clave y volvé a ejecutar ***")
        break

    V = todo[todo["ok"] == True].merge(
        Q[["spec_id", "tipo", "serie_id", "coloquial", "tecnica"]], on="spec_id")
    print(f"  Verificadas: {len(V)} de {len(Q)}")
    print(f"  Marcadas — coloquial: {(~V['col_ok']).mean():.1%} | técnica: {(~V['tec_ok']).mean():.1%}")
    print("  Coloquial marcada por tipo:", V.groupby("tipo")["col_ok"].apply(lambda s: round((~s).mean(), 3)).to_dict())
    for _, r in V[~V["col_ok"]].head(6).iterrows():
        print(f"   * [{r['tipo']}] {r['coloquial']}  |  motivo: {r['col_motivo'][:110]}")
    V.to_csv(f"{RUTA}/verificacion_fidelidad_{split}.csv", index=False)
    print(f"  Guardado: verificacion_fidelidad_{split}.csv")

In [ ]:
print("CELDA: ARMADO FINAL DE VAL Y TEST CON MARCAS DE FIDELIDAD (SIN COSTO)")
from google.colab import drive
import pandas as pd

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"

for split in ["val", "test"]:
    Q = pd.read_csv(f"{RUTA}/preguntas_{split}_v2.csv")
    V = pd.read_csv(f"{RUTA}/verificacion_fidelidad_{split}.csv")[
        ["spec_id", "col_ok", "col_motivo", "tec_ok", "tec_motivo"]]
    M = Q.merge(V, on="spec_id", how="left")
    assert M["col_ok"].notna().all() and M["tec_ok"].notna().all(), "Faltan verificaciones"
    M["col_ok"], M["tec_ok"] = M["col_ok"].astype(bool), M["tec_ok"].astype(bool)

    clave = M["coloquial"].str.lower().str.strip()
    M["dup_coloquial"] = clave.duplicated(keep=False)
    M["excluida"] = ~M["tec_ok"]
    M["coloquial_ambigua"] = (~M["col_ok"]) | M["dup_coloquial"]
    F = M[~M["excluida"]].copy()

    print(f"\n=== {split}: {len(M)} preguntas | excluidas (técnica marcada): {int(M['excluida'].sum())} "
          f"| quedan: {len(F)} (series {F['serie_id'].nunique()}, familias {F['familia'].nunique()})")
    print(f"  coloquial ambigua entre las que quedan: {int(F['coloquial_ambigua'].sum())} "
          f"({F['coloquial_ambigua'].mean():.1%}) | de ellas, repetidas: {int(F['dup_coloquial'].sum())}")
    print("  por tipo (total / ambigua):")
    print(F.groupby("tipo")["coloquial_ambigua"].agg(total="size", ambigua="sum").to_string())
    exc = M[M["excluida"]]
    print("  familias de las excluidas:", exc["familia"].value_counts().head(3).to_dict())

    print("  Motivos de coloquiales marcadas (10 al azar, completos):")
    for _, r in M[~M["col_ok"]].sample(min(10, int((~M["col_ok"]).sum())), random_state=1).iterrows():
        print(f"   * [{r['tipo']}] {r['coloquial']}\n       -> {str(r['col_motivo'])[:220]}")

    F.to_csv(f"{RUTA}/eval_{split}_final.csv", index=False)
    exc.to_csv(f"{RUTA}/eval_{split}_excluidas.csv", index=False)
    print(f"  Guardado: eval_{split}_final.csv y eval_{split}_excluidas.csv")

In [ ]:
!pip -q install -U anthropic
import os, re, json, time, threading
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import anthropic
from google.colab import drive, userdata

print("CELDA: VERIFICADOR DE FIDELIDAD (TRAIN)")
drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
SPLIT = "train"
HILOS = 4
MODEL = "claude-haiku-5-5"
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=3, timeout=60.0)
ABORT = threading.Event()

cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id"))

def g(c, col):
    v = c[col]
    return "" if pd.isna(v) else str(v)

MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "octubre", "noviembre", "diciembre"]
TRIM = ["primer", "segundo", "tercer", "cuarto"]

def periodo_de_spec(r):
    if r["tipo"] == "ultimo_dato":
        return None
    p = json.loads(r["params"])
    if r["tipo"] == "variacion_interanual":
        e = p["end_date"]
        return e[:4] if r["frecuencia"] == "anual" else e[:7]
    return p["start_date"]

def periodo_humano(freq, p):
    if freq == "anual":
        return p[:4]
    y, m = p[:4], int(p[5:7])
    return f"{MESES[m - 1]} de {y}" if freq == "mensual" else f"{TRIM[(m - 1) // 3]} trimestre de {y}"

def operacion(r):
    p = periodo_de_spec(r)
    if r["tipo"] == "ultimo_dato":
        return "el dato más reciente disponible"
    hp = periodo_humano(r["frecuencia"], p)
    if r["tipo"] == "variacion_interanual":
        return f"la variación interanual en {hp}"
    return f"el valor en {hp}"

def armar_prompt(r):
    c = cat.loc[r["serie_id"]]
    return f"""Verificá si cada pregunta pide el dato descrito abajo, o si cambia su sentido (otra medida, otra entidad, otra operación u otro período).
Una pregunta coloquial puede ser más vaga que el dato, pero tiene que poder referirse razonablemente a ESE dato y no a uno claramente distinto.
Marcá "ok": false solo si hay un cambio de sentido real.

Dato real:
- Título técnico: {g(c, 'serie_titulo')}
- Descripción: {g(c, 'serie_descripcion')}
- Unidades: {g(c, 'serie_unidades')}
- Frecuencia: {g(c, 'frecuencia')}
- Dataset: {g(c, 'dataset_titulo')}
- Distribución: {g(c, 'distribucion_titulo')}
- Lo que se pide: {operacion(r)}

Pregunta coloquial: {r['coloquial']}
Pregunta técnica: {r['tecnica']}

Devolvé solo JSON: {{"coloquial": {{"ok": true, "motivo": ""}}, "tecnica": {{"ok": true, "motivo": ""}}}}"""

def tarea(r):
    if ABORT.is_set():
        return {"spec_id": r["spec_id"], "ok": False, "error": "abortado"}
    try:
        msg = client.messages.create(model=MODEL, max_tokens=500,
                                     messages=[{"role": "user", "content": armar_prompt(r)}],
                                     extra_body={"thinking": {"type": "disabled"}})
        txt = "".join(b.text for b in msg.content if getattr(b, "type", "") == "text")
        js = json.loads(txt[txt.find("{"): txt.rfind("}") + 1])
        return {"spec_id": r["spec_id"], "ok": True, "error": "",
                "col_ok": bool(js["coloquial"]["ok"]), "col_motivo": str(js["coloquial"].get("motivo", "")),
                "tec_ok": bool(js["tecnica"]["ok"]), "tec_motivo": str(js["tecnica"].get("motivo", "")),
                "ent": msg.usage.input_tokens, "sal": msg.usage.output_tokens}
    except anthropic.AuthenticationError as e:
        ABORT.set()
        return {"spec_id": r["spec_id"], "ok": False, "error": f"AuthenticationError (se corta todo): {str(e)[:80]}"}
    except Exception as e:
        return {"spec_id": r["spec_id"], "ok": False, "error": f"{type(e).__name__}: {str(e)[:100]}"}

parcial = f"{RUTA}/verificacion_fidelidad_{SPLIT}_parcial.csv"
Q = pd.read_csv(f"{RUTA}/preguntas_{SPLIT}_v1.csv")
dup = Q["coloquial"].str.lower().value_counts()
print(f"\n=== {SPLIT}: {len(Q)} especificaciones | coloquiales repetidas: {int((dup > 1).sum())}")

prev = pd.read_csv(parcial) if os.path.exists(parcial) else pd.DataFrame()
if len(prev):
    prev = prev[prev["ok"] == True]
hechos = set(prev["spec_id"]) if len(prev) else set()
pend = [r for r in Q.to_dict("records") if r["spec_id"] not in hechos]
print(f"  ya hechas: {len(hechos)} | pendientes: {len(pend)}")

t0, res = time.time(), []
with ThreadPoolExecutor(max_workers=HILOS) as ex:
    for k, out in enumerate(ex.map(tarea, pend), 1):
        res.append(out)
        if k % 100 == 0:
            pd.concat([prev, pd.DataFrame(res)], ignore_index=True).to_csv(parcial, index=False)
            print(f"  [{k}/{len(pend)}] {time.time() - t0:.0f}s", flush=True)
R = pd.DataFrame(res)
todo = pd.concat([prev, R], ignore_index=True) if len(R) else prev
todo.to_csv(parcial, index=False)

if len(R):
    costo = R["ent"].fillna(0).sum() * 0.10 / 1e6 + R["sal"].fillna(0).sum() * 0.50 / 1e6
    print(f"  Esta corrida: {int(R['ok'].sum())}/{len(R)} ok | costo ${costo:.4f} | {time.time() - t0:.0f}s")
    print("  Fallos:", Counter(R[~R['ok']]["error"]).most_common(3))
if ABORT.is_set():
    print("  *** Se cortó por error de autenticación: revisá la clave y volvé a ejecutar ***")
else:
    V = todo[todo["ok"] == True].merge(
        Q[["spec_id", "tipo", "serie_id", "familia", "coloquial", "tecnica"]], on="spec_id")
    print(f"  Verificadas: {len(V)} de {len(Q)}")
    print(f"  Marcadas — coloquial: {(~V['col_ok']).mean():.1%} | técnica: {(~V['tec_ok']).mean():.1%}")
    print("  Coloquial marcada por tipo:", V.groupby("tipo")["col_ok"].apply(lambda s: round((~s).mean(), 3)).to_dict())
    print("  Técnica marcada por tipo:", V.groupby("tipo")["tec_ok"].apply(lambda s: round((~s).mean(), 3)).to_dict())
    print("  Familias con más técnicas marcadas:", V[~V["tec_ok"]]["familia"].value_counts().head(3).to_dict())
    V.to_csv(f"{RUTA}/verificacion_fidelidad_{SPLIT}.csv", index=False)
    print(f"  Guardado: verificacion_fidelidad_{SPLIT}.csv")

In [ ]:
print("CELDA: ARMADO FINAL DE TRAIN CON MARCAS DE FIDELIDAD (SIN COSTO)")
from google.colab import drive
import pandas as pd

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"

Q = pd.read_csv(f"{RUTA}/preguntas_train_v1.csv")
P = pd.read_csv(f"{RUTA}/verificacion_fidelidad_train_parcial.csv")
P = P[P["ok"] == True].drop_duplicates("spec_id", keep="last").copy()
for col in ["col_ok", "tec_ok"]:
    P[col] = P[col].astype(str).str.lower().eq("true")      # evita el problema de los -1 y -2

M = Q.merge(P[["spec_id", "col_ok", "col_motivo", "tec_ok", "tec_motivo"]], on="spec_id", how="left")
M["verificada"] = M["spec_id"].isin(P["spec_id"])
M["col_ok"] = M["col_ok"].fillna(False).astype(bool)
M["tec_ok"] = M["tec_ok"].fillna(False).astype(bool)

clave = M["coloquial"].str.lower().str.strip()
M["dup_coloquial"] = clave.duplicated(keep=False)
M["excluida"] = ~M["verificada"] | ~M["tec_ok"]
M["coloquial_ambigua"] = ~M["col_ok"] | M["dup_coloquial"]
V = M[M["verificada"]]
F = M[~M["excluida"]].copy()

print(f"Preguntas: {len(M)} | verificadas: {int(M['verificada'].sum())} | sin verificar: {int((~M['verificada']).sum())}")
print(f"Marcadas — coloquial: {(~V['col_ok']).mean():.1%} | técnica: {(~V['tec_ok']).mean():.1%}")
print("Coloquial marcada por tipo:", V.groupby("tipo")["col_ok"].apply(lambda s: round((~s).mean(), 3)).to_dict())
print("Técnica marcada por tipo:", V.groupby("tipo")["tec_ok"].apply(lambda s: round((~s).mean(), 3)).to_dict())
print("Familias con más técnicas marcadas:", V[~V["tec_ok"]]["familia"].value_counts().head(3).to_dict())

print(f"\nQuedan: {len(F)} (series {F['serie_id'].nunique()}, familias {F['familia'].nunique()}) | excluidas: {int(M['excluida'].sum())}")
print("Por tipo:", F["tipo"].value_counts().to_dict())
print("% de sin_datos entre las que quedan:", f"{(F['tipo'] == 'sin_datos').mean():.1%}")
usable_col = F[~F["coloquial_ambigua"]]
print(f"Coloquiales utilizables para entrenar: {len(usable_col)} ({len(usable_col) / len(F):.1%}) | por tipo:",
      usable_col["tipo"].value_counts().to_dict())
print(f"Coloquiales repetidas: {int(M['dup_coloquial'].sum())} filas")

print("\nTécnicas marcadas (10 al azar, motivo completo):")
T = V[~V["tec_ok"]]
for _, r in T.sample(min(10, len(T)), random_state=1).iterrows():
    print(f"  * [{r['tipo']}] {r['tecnica']}\n      -> {str(r['tec_motivo'])[:240]}")

F.to_csv(f"{RUTA}/train_final.csv", index=False)
print("\nGuardado: train_final.csv")

In [ ]:
print("CELDA: DIAGNÓSTICO DE DEFECTOS NO DETECTADOS (SIN COSTO)")
import re, unicodedata
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
APLICAR = False     # True: guarda copias nuevas sin los casos con defecto (no pisa nada)

SALIDAS = {"train": "train_final.csv", "val": "eval_val_final.csv", "test": "eval_test_final.csv"}
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "setiembre", "octubre", "noviembre", "diciembre"]
re_mes = re.compile(r"\b(" + "|".join(MESES) + r")\b")
re_trim = re.compile(r"\btrimestre\b")
re_tasa_titulo = re.compile(r"(^|_)(var|variacion|tasa|pct|porcentaje)(_|$)")
re_tasa_unidad = re.compile(r"%|porcent|variaci")

def minusculas(t):
    return unicodedata.normalize("NFKD", str(t).lower()).encode("ascii", "ignore").decode()

def defecto_periodo(freq, tipo, texto):
    t = minusculas(texto)
    if tipo == "ultimo_dato" and re_mes.search(t):
        return "fecha en pregunta de último dato"
    if freq == "anual" and (re_mes.search(t) or re_trim.search(t)):
        return "mes o trimestre en serie anual"
    if freq == "trimestral" and re_mes.search(t):
        return "mes en serie trimestral"
    if freq == "mensual" and re_trim.search(t):
        return "trimestre en serie mensual"
    return ""

cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id")[["serie_titulo", "serie_unidades"]])

def primer_defecto(r):
    for txt in (r["coloquial"], r["tecnica"]):
        m = defecto_periodo(r["frecuencia"], r["tipo"], txt)
        if m:
            return m
    return ""

for split, arch in SALIDAS.items():
    F = pd.read_csv(f"{RUTA}/{arch}").merge(cat, left_on="serie_id", right_index=True, how="left")
    F["def_periodo"] = F.apply(primer_defecto, axis=1).fillna("")
    es_tasa = (F["serie_titulo"].map(lambda s: bool(re_tasa_titulo.search(str(s).lower())))
               | F["serie_unidades"].map(lambda u: bool(re_tasa_unidad.search(str(u).lower()))))
    F["def_tasa"] = (F["tipo"] == "variacion_interanual") & es_tasa
    mal = (F["def_periodo"] != "") | F["def_tasa"]

    n_var = int((F["tipo"] == "variacion_interanual").sum())
    print(f"\n=== {split}: {len(F)} preguntas")
    print("  Defecto de período:", F[F["def_periodo"] != ""]["def_periodo"].value_counts().to_dict())
    print(f"  Variación interanual sobre series que ya son tasas o variaciones: {int(F['def_tasa'].sum())} de {n_var}")
    print(f"  Total con algún defecto: {int(mal.sum())} ({mal.mean():.1%})")
    for _, r in F[F["def_periodo"] != ""].head(3).iterrows():
        print(f"   [período] [{r['tipo']}|{r['frecuencia']}] {r['coloquial']}  //  {r['tecnica'][:110]}")
    for _, r in F[F["def_tasa"]].head(5).iterrows():
        print(f"   [tasa] {r['serie_titulo']} | unidades: {r['serie_unidades']}")
    if APLICAR:
        nombre = arch.replace("_final.csv", "_final2.csv")
        out = F[~mal].drop(columns=["def_periodo", "def_tasa", "serie_titulo", "serie_unidades"])
        out.to_csv(f"{RUTA}/{nombre}", index=False)
        print(f"  Guardado: {nombre} ({len(out)} preguntas)")

if not APLICAR:
    print("\nAPLICAR = False: no se guardó nada")

In [ ]:
print("CELDA: APLICAR CORRECCIONES DE DEFECTOS (SIN COSTO)")
import re, unicodedata
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
APLICAR = False     # True: guarda los *_final2.csv (no pisa los originales)

SALIDAS = {"train": "train_final.csv", "val": "eval_val_final.csv", "test": "eval_test_final.csv"}
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "setiembre", "octubre", "noviembre", "diciembre"]
re_mes = re.compile(r"\b(" + "|".join(MESES) + r")\b")
re_trim = re.compile(r"\btrimestre\b")
re_tasa_unidad = re.compile(r"%|porcent|variaci")
re_var_titulo = re.compile(r"(^|_)(var|variacion|ia)(_|$)")

def minusculas(t):
    return unicodedata.normalize("NFKD", str(t).lower()).encode("ascii", "ignore").decode()

def defecto_periodo(freq, tipo, texto):
    t = minusculas(texto)
    if tipo == "ultimo_dato" and re_mes.search(t):
        return "fecha en pregunta de último dato"
    if freq == "anual" and (re_mes.search(t) or re_trim.search(t)):
        return "mes o trimestre en serie anual"
    if freq == "trimestral" and re_mes.search(t):
        return "mes en serie trimestral"
    if freq == "mensual" and re_trim.search(t):
        return "trimestre en serie mensual"
    return ""

cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id")[["serie_titulo", "serie_unidades"]])

def marca_var(r):
    if r["tipo"] != "variacion_interanual":
        return ""
    tit, uni = str(r["serie_titulo"]).lower(), str(r["serie_unidades"]).lower()
    if re_var_titulo.search(tit) or "variaci" in uni:
        return "var_sobre_tasa"
    if re_tasa_unidad.search(uni) or "tasa" in tit:
        return "var_sobre_porcentaje"
    return ""

for split, arch in SALIDAS.items():
    F = pd.read_csv(f"{RUTA}/{arch}").merge(cat, left_on="serie_id", right_index=True, how="left")
    F["def_periodo"] = [defecto_periodo(f, t, c) or defecto_periodo(f, t, te)
                        for f, t, c, te in zip(F["frecuencia"], F["tipo"], F["coloquial"], F["tecnica"])]
    F["marca_var"] = F.apply(marca_var, axis=1)
    excl = F["def_periodo"] != ""
    out = F[~excl].copy()
    print(f"\n=== {split}: {len(F)} -> {len(out)} (se excluyen {int(excl.sum())} por defecto de período)")
    print("  marca_var:", out["marca_var"].value_counts().to_dict())
    print("  por tipo:", out["tipo"].value_counts().to_dict())
    print("  % sin_datos:", f"{(out['tipo'] == 'sin_datos').mean():.1%}")
    ej = out[out["marca_var"] == "var_sobre_tasa"].head(4)
    for _, r in ej.iterrows():
        print(f"   [var_sobre_tasa] {r['serie_titulo']} | {r['serie_unidades']}")
    ej = out[out["marca_var"] == "var_sobre_porcentaje"].head(4)
    for _, r in ej.iterrows():
        print(f"   [var_sobre_porcentaje] {r['serie_titulo']} | {r['serie_unidades']}")
    if APLICAR:
        nombre = arch.replace("_final.csv", "_final2.csv")
        out.drop(columns=["def_periodo", "serie_titulo", "serie_unidades"]).to_csv(f"{RUTA}/{nombre}", index=False)
        print(f"  Guardado: {nombre}")

if not APLICAR:
    print("\nAPLICAR = False: no se guardó nada")

In [ ]:
import os
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
for a in ["train_final2.csv", "eval_val_final2.csv", "eval_test_final2.csv"]:
    print(a, "-> existe" if os.path.exists(f"{RUTA}/{a}") else "-> FALTA")

In [ ]:
print("CELDA: APLICAR CORRECCIONES DE DEFECTOS (GUARDA LOS final2)")
import re, unicodedata
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
APLICAR = True      # guarda los *_final2.csv (no pisa los originales)

SALIDAS = {"train": "train_final.csv", "val": "eval_val_final.csv", "test": "eval_test_final.csv"}
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "setiembre", "octubre", "noviembre", "diciembre"]
re_mes = re.compile(r"\b(" + "|".join(MESES) + r")\b")
re_trim = re.compile(r"\btrimestre\b")
re_tasa_unidad = re.compile(r"%|porcent|variaci")
re_var_titulo = re.compile(r"(^|_)(var|variacion|ia)(_|$)")

def minusculas(t):
    return unicodedata.normalize("NFKD", str(t).lower()).encode("ascii", "ignore").decode()

def defecto_periodo(freq, tipo, texto):
    t = minusculas(texto)
    if tipo == "ultimo_dato" and re_mes.search(t):
        return "fecha en pregunta de último dato"
    if freq == "anual" and (re_mes.search(t) or re_trim.search(t)):
        return "mes o trimestre en serie anual"
    if freq == "trimestral" and re_mes.search(t):
        return "mes en serie trimestral"
    if freq == "mensual" and re_trim.search(t):
        return "trimestre en serie mensual"
    return ""

cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id")[["serie_titulo", "serie_unidades"]])

def marca_var(r):
    if r["tipo"] != "variacion_interanual":
        return ""
    tit, uni = str(r["serie_titulo"]).lower(), str(r["serie_unidades"]).lower()
    if re_var_titulo.search(tit) or "variaci" in uni:
        return "var_sobre_tasa"
    if re_tasa_unidad.search(uni) or "tasa" in tit:
        return "var_sobre_porcentaje"
    return ""

for split, arch in SALIDAS.items():
    F = pd.read_csv(f"{RUTA}/{arch}").merge(cat, left_on="serie_id", right_index=True, how="left")
    F["def_periodo"] = [defecto_periodo(f, t, c) or defecto_periodo(f, t, te)
                        for f, t, c, te in zip(F["frecuencia"], F["tipo"], F["coloquial"], F["tecnica"])]
    F["marca_var"] = F.apply(marca_var, axis=1)
    excl = F["def_periodo"] != ""
    out = F[~excl].copy()
    print(f"\n=== {split}: {len(F)} -> {len(out)} (se excluyen {int(excl.sum())} por defecto de período)")
    print("  marca_var:", out["marca_var"].value_counts().to_dict())
    print("  por tipo:", out["tipo"].value_counts().to_dict())
    if APLICAR:
        nombre = arch.replace("_final.csv", "_final2.csv")
        out.drop(columns=["def_periodo", "serie_titulo", "serie_unidades"]).to_csv(f"{RUTA}/{nombre}", index=False)
        print(f"  Guardado: {nombre}")

if not APLICAR:
    print("\nAPLICAR = False: no se guardó nada")